# Taobao Night Browsing — Full Pipeline (single file)

This notebook runs the **entire retained project** end-to-end from a single data dependency: the raw `UserBehavior.csv` downloaded from the public mirror. Execution order:

**Download → clean → model / experiment → figures**.

- **Data**: Tianchi 649 (Taobao UserBehavior) public mirror, 100,150,807 raw events → 100,095,182 cleaned events, Asia/Shanghai 2017-11-25 to 12-03 (9 days).
- **Unit of analysis**: user–item "first-visible-view" episode, 51,316,512 episodes across 983,132 users.
- **Question**: is late-night browsing (22–01) "impulse buying" or "idle browsing"?

## Pipeline overview

| Step | What | Output | Time (full) |
|---|---|---|---|
| 1 | Point to the local raw `UserBehavior.csv` | — | manual |
| 2 | Clean raw CSV (time window + dedup + behavior code) | `data/processed/events_clean.parquet`, `user_buckets.parquet` | ~5 min |
| 3 | events → episodes | `data/episodes_hour.parquet` | ~6.5 min |
| 4 | Fixed-effects LPM (M0/M1/M2 + robustness + hour shape) | `results/model_coefficients.csv` etc. | ~27 min |
| 5 | Independent verification (rebuild + FE vs explicit-dummy OLS) | `results/verification.json` | ~2 min |
| 6 | Pre-t0 features + independent recount | `data/episodes_features.parquet` | ~2 min |
| 7 | Survival analysis (Cox M1–M3 / piecewise / Weibull) | `results/survival_*.csv` | ~12 min / 17 GB |
| 8 | Causal Forest (ATE / BLP / GATES) | `results/causal_forest_full.json` | ~51 min / 22 GB |
| 9 | Phase 2 A: survival by hour × three windows | `results/survival_hourly_*.csv` | ~8 min |
| 10 | Phase 2 B: cold vs hot items | `results/cold_hot_items_*.csv` | ~3 min |
| 11 | Category extension: survival by category | `results/category_survival_*.csv` | ~8 min |
| 12 | Category extension: Causal Forest with category dummies | `results/causal_forest_category_full.json` | ~60 min |
| 13–16 | Figures 01–11 | `figures/01–11` | <1 min each |

## Environment

- Python 3.13.9; set `OPENBLAS_NUM_THREADS=2 OMP_NUM_THREADS=2` before running.
- Dependencies: numpy / pandas / scipy / statsmodels / matplotlib / `tqdm` + `pyhdfe` / `lightgbm` / `econml` / `lifelines` (installed via `requirements.txt`).
- The **only data dependency** is the raw CSV (local path set in step 1); every downstream table is regenerated.

## Before re-running

- Survival (full) ~12 min / 17 GB; Causal Forest (full) ~51 min / 22 GB. Causal Forest requires `max_samples=0.05` (the econml default 0.45 pre-allocates ~70 GB and OOMs).
- Each heavy script uses a `TAG` switch: `'smoke'` = quick small-sample check; `'full'` = full run (matches the saved results).
- Step 2 (cleaning) is skipped when its outputs already exist locally.
- Progress bars (`tqdm`) report cleaning and per-unit experiment loops.

> Convention: code comments and docstrings are English; the Markdown narrative is English too.


## 0 · Environment and shared utilities

Path constants, logging, `pyhdfe` fixed-effect absorption, `_Args` (a stand-in for argparse so scripts can be configured inline), and the global `TAG` switch.


In [13]:
from pathlib import Path
import sys, time, json, warnings, hashlib
import numpy as np, pandas as pd
from scipy import stats
from tqdm.auto import tqdm   # progress bars

# Project root: locate it by walking up from the current directory (works no matter where Jupyter starts).
def _find_root(start):
    for p in [start] + list(start.parents):
        if (p / "requirements.txt").exists() and (p / "notebooks").is_dir():
            return p
    return start
EXT = _find_root(Path.cwd())
DATA = EXT / "data"                         # raw download + episode tables
PROC = DATA / "processed"                   # cleaned events + user md5 buckets
RES  = EXT / "results"                      # result CSV/JSON
FIG  = EXT / "figures"                      # figures
for p in (DATA, PROC, RES, FIG):
    p.mkdir(parents=True, exist_ok=True)

import pyhdfe                              # high-dimensional fixed-effect absorption (requirements.txt)

warnings.filterwarnings("ignore")

def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)

class _Args:
    """Stand-in for argparse so the heavy scripts can be configured inline."""
    def __init__(self, **kw):
        self.__dict__.update(kw)

# ---- global run toggle ----
TAG = "smoke"   # "smoke" = quick small-sample check; "full" = full run (matches saved results)


## 0.1 · Two-way clustered covariance (Cameron–Gelbach–Miller)

Cluster-robust covariance for user × item or user × dayhour, with PSD repair.


In [14]:
def cluster_cov(X, e, groups, k_total):
    n, k = X.shape
    bread = np.linalg.inv(X.T @ X)
    score = X * e[:, None]

    def meat(g):
        code = pd.factorize(g, sort=False)[0]; G = int(code.max() + 1)
        sums = np.column_stack([np.bincount(code, weights=score[:, j], minlength=G) for j in range(k)])
        return sums.T @ sums * (G / (G - 1)) * ((n - 1) / (n - k_total)), G

    M, G = meat(groups[0]); counts = [G]
    if len(groups) == 2:
        M2, G2 = meat(groups[1])
        inter = groups[0].astype(np.int64) * (1 << 32) + groups[1].astype(np.int64)
        M12, G12 = meat(inter)
        M = M + M2 - M12; counts += [G2, G12]
    cov = bread @ M @ bread; cov = (cov + cov.T) / 2
    eig = np.linalg.eigvalsh(cov)
    if eig.min() < 0:  # standard PSD repair for two-way clustering
        w, v = np.linalg.eigh(cov); cov = v @ np.diag(np.clip(w, 0, None)) @ v.T
    return cov, counts, float(eig.min())


## 0.2 · FastCox (hand-written Cox proportional hazards)

statsmodels / lifelines are infeasible on 30.8 M rows, so a hand-written solver is used: sort once by (stratum, time desc), risk sets via group-wise cumulative sums; Hessian via `sum_k d_k S2_k / S0_k = X' diag(w * C_A) X`; Lin-Wei clustered variance. Verified against statsmodels / lifelines in step 5.2.


In [15]:
class FastCox:
    def __init__(self, time, event, X, strata=None):
        n = len(time)
        s = np.zeros(n, np.int64) if strata is None else pd.factorize(strata)[0].astype(np.int64)
        order = np.lexsort((-np.asarray(time, np.float64), s))
        self.order = order
        self.t = np.asarray(time, np.float64)[order]
        self.e = np.asarray(event, np.float64)[order]
        self.X = np.ascontiguousarray(np.asarray(X, np.float64)[order])
        self.s = s[order]
        new = np.ones(n, bool)
        new[1:] = (self.s[1:] != self.s[:-1]) | (self.t[1:] != self.t[:-1])
        self.gstart = np.flatnonzero(new)
        self.gid = np.cumsum(new) - 1
        self.gs = self.s[self.gstart]
        self.D = np.add.reduceat(self.e, self.gstart)
        self.SXe = np.add.reduceat(self.X * self.e[:, None], self.gstart)
        sn = np.ones(len(self.gstart), bool); sn[1:] = self.gs[1:] != self.gs[:-1]
        self.sstart = np.flatnonzero(sn)

    def _cum_within(self, a, reverse=False):
        """Cumulative sum over groups, reset at stratum boundaries (forward = descending time)."""
        a2 = a if a.ndim == 2 else a[:, None]
        if reverse:
            a2 = a2[::-1]
            starts = len(a) - 1 - np.r_[self.sstart[1:] - 1, len(a) - 1][::-1]
        else:
            starts = self.sstart
        c = np.cumsum(a2, axis=0)
        seg = np.repeat(np.arange(len(starts)), np.diff(np.r_[starts, len(a)]))
        off = np.vstack([np.zeros((1, a2.shape[1])), c[starts[1:] - 1]]) if len(starts) > 1 else np.zeros((1, a2.shape[1]))
        out = c - off[seg]
        if reverse:
            out = out[::-1]
        return out if a.ndim == 2 else out[:, 0]

    def _parts(self, beta):
        eta = self.X @ beta
        c = eta.max(); w = np.exp(eta - c)
        R0 = self._cum_within(np.add.reduceat(w, self.gstart))
        R1 = self._cum_within(np.add.reduceat(self.X * w[:, None], self.gstart))
        ev = self.D > 0
        ll = float(self.SXe.sum(0) @ beta - np.sum(self.D[ev] * (np.log(R0[ev]) + c)))
        xbar = R1 / R0[:, None]
        grad = self.SXe.sum(0) - (self.D[:, None] * xbar).sum(0)
        A = np.where(ev, self.D / R0, 0.0)
        CA = self._cum_within(A, reverse=True)[self.gid]
        H = -(self.X.T @ (self.X * (w * CA)[:, None]) - (xbar * self.D[:, None]).T @ xbar)
        return ll, grad, H, w, xbar, A

    def fit(self, max_iter=30, tol=1e-9):
        p = self.X.shape[1]; beta = np.zeros(p)
        ll, g, H, *_ = self._parts(beta)
        for it in range(max_iter):
            step = np.linalg.solve(-H, g)
            lam = 1.0
            while True:
                nb = beta + lam * step
                nll, ng, nH, *_ = self._parts(nb)
                if (np.isfinite(nll) and nll >= ll - 1e-12) or lam < 1e-6:
                    break
                lam /= 2
            done = abs(nll - ll) < tol * (1 + abs(ll))
            beta, ll, g, H = nb, nll, ng, nH
            if done:
                break
        self.beta, self.loglik, self.n_iter, self.H = beta, ll, it + 1, H
        self.converged = bool(done and np.all(np.isfinite(beta)))
        if not self.converged:
            raise RuntimeError(f"FastCox did not converge after {it + 1} iterations (loglik {ll})")
        return self

    def robust_cov(self, groups):
        """Lin-Wei sandwich, clustered by `groups` (original row order), CR1 small-sample factor."""
        ll, g, H, w, xbar, A = self._parts(self.beta)
        CA = self._cum_within(A, reverse=True)[self.gid]
        CB = self._cum_within(A[:, None] * xbar, reverse=True)[self.gid]
        U = self.e[:, None] * (self.X - xbar[self.gid]) - w[:, None] * (self.X * CA[:, None] - CB)
        code = pd.factorize(np.asarray(groups)[self.order])[0]; G = code.max() + 1
        S = np.column_stack([np.bincount(code, weights=U[:, j], minlength=G) for j in range(U.shape[1])])
        Hi = np.linalg.inv(-H)
        self.cov = Hi @ (S.T @ S) @ Hi * (G / (G - 1)); self.n_clusters = int(G)
        self.naive_cov = Hi
        return self.cov

    def summary(self, names):
        se = np.sqrt(np.diag(self.cov)); z = self.beta / se
        return pd.DataFrame({"term": names, "coef": self.beta, "se": se, "hr": np.exp(self.beta),
                             "hr_ci_low": np.exp(self.beta - 1.96 * se), "hr_ci_high": np.exp(self.beta + 1.96 * se),
                             "p": 2 * stats.norm.sf(np.abs(z))})


## 0.3 · Shared helpers for the advanced methods (adv_common)

Loading by user md5 bucket, time bins, pre-t0 features, one-way cluster-robust OLS — shared by the survival and Causal Forest steps.


In [16]:
import pyarrow.dataset as ds

LATE = [22, 23, 0, 1]
DAYH = list(range(9, 18))
BINS = {"day": DAYH, "evening": [18, 19, 20, 21], "late": LATE, "small": [2, 3, 4, 5], "morning": [6, 7, 8]}
WEEKEND_DAYS = (1, 7, 8)  # day 0 = Sat 2017-11-25
CF_FEATURES = ["l_hpv24", "l_hpv1h", "hist_buy24", "l_pop24", "l_upv", "l_ufav", "l_ucart", "l_ubuy", "l_ibuy24",
               "day", "is_weekend"]

def parse_range(s):
    lo, hi = map(int, s.split("-"))
    return lo, hi

def load(ranges, hours=None):
    """Episodes whose user bucket falls in any inclusive (lo, hi) range; optional t0-hour filter."""
    f = None
    for lo, hi in ranges:
        g = (ds.field("bucket") >= lo) & (ds.field("bucket") <= hi)
        f = g if f is None else (f | g)
    if hours is not None:
        f = f & ds.field("hour").isin(hours)
    d = ds.dataset(DATA / "episodes_features.parquet").to_table(filter=f).to_pandas()
    dl = d.buy_delay_s.to_numpy()
    d["y1"] = ((dl >= 0) & (dl < 3600)).astype(np.float64)
    d["y24"] = ((dl >= 0) & (dl < 86400)).astype(np.float64)
    d["l_hpv24"] = np.log1p(d.hist_pv24); d["l_hpv1h"] = np.log1p(d.hist_pv1h); d["l_pop24"] = np.log1p(d.pop24)
    d["l_upv"] = np.log1p(d.u_pv_cum); d["l_ufav"] = np.log1p(d.u_fav_cum); d["l_ucart"] = np.log1p(d.u_cart_cum)
    d["l_ubuy"] = np.log1p(d.u_buy_cum); d["l_ibuy24"] = np.log1p(d.i_buy24)
    d["hist_buy24"] = d.hist_buy24.astype(np.float64)
    d["day"] = d.day.astype(np.int64)
    d["is_weekend"] = d.day.isin(WEEKEND_DAYS).astype(np.float64)
    hb = np.empty(24, dtype=object)
    for name, hs in BINS.items():
        hb[hs] = name
    d["bin"] = hb[d.hour.to_numpy()]
    return d

def split_name(bucket, train, val, test):
    out = np.full(len(bucket), "", dtype=object)
    for name, (lo, hi) in [("train", train), ("val", val), ("test", test)]:
        out[(bucket >= lo) & (bucket <= hi)] = name
    return out

def cluster_ols(y, X, groups):
    """OLS with one-way cluster-robust covariance (CR1). Returns beta, cov, G."""
    XtX_inv = np.linalg.inv(X.T @ X)
    beta = XtX_inv @ (X.T @ y)
    e = y - X @ beta
    code = pd.factorize(groups)[0]; G = code.max() + 1
    S = np.column_stack([np.bincount(code, weights=X[:, j] * e, minlength=G) for j in range(X.shape[1])])
    n, k = X.shape
    cov = XtX_inv @ (S.T @ S) @ XtX_inv * (G / (G - 1)) * ((n - 1) / (n - k))
    return beta, cov, int(G)


## 0.4 · Shared figure style (fig_style)

Maroon #7C3A47 + warm grey palette, matching the mid-term deck; all figures share this style.


In [17]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

MAROON = "#7C3A47"
MAROON_MID = "#A85D68"
ROSE = "#C98A95"
PINK = "#D8A7AF"
PALE = "#E7DAE0"
DARK = "#403737"
CHAR = "#4A4545"
GRAY = "#9A8F8F"

def apply():
    plt.rcParams.update({
        "figure.facecolor": "white", "axes.facecolor": "white",
        "axes.edgecolor": "#B9ABAE", "axes.linewidth": 0.8,
        "axes.labelcolor": CHAR, "axes.titlecolor": DARK,
        "axes.titlesize": 10, "axes.labelsize": 9,
        "xtick.color": CHAR, "ytick.color": CHAR,
        "xtick.labelsize": 8, "ytick.labelsize": 8,
        "grid.color": "#EFE7E9", "grid.linewidth": 0.7,
        "font.size": 9, "legend.frameon": False, "legend.fontsize": 8,
        "axes.spines.top": False, "axes.spines.right": False,
        "figure.dpi": 150,
    })


## 1 · Set the path to the raw dataset

Set `RAW_CSV_PATH` in the cell below to your **local copy** of the raw `UserBehavior.csv` (100,150,807 rows, ~3.67 GB; see `data/source_manifest.json` for where to obtain it). Everything downstream is regenerated from this single file — no other data is needed.


In [18]:
# ---- step 1: point to the raw dataset (EDIT THIS CELL) ----
# Set RAW_CSV_PATH to your local UserBehavior.csv. Examples:
#   macOS / Linux :  RAW_CSV_PATH = "/Users/you/Downloads/UserBehavior.csv"
#   Windows       :  RAW_CSV_PATH = r"C:\Users\you\Downloads\UserBehavior.csv"
RAW_CSV_PATH = str(EXT / "data" / "UserBehavior.csv")

from pathlib import Path as _Path
if not (RAW_CSV_PATH and _Path(RAW_CSV_PATH).expanduser().is_file()):
    raise FileNotFoundError(
        f"raw CSV not found: {RAW_CSV_PATH!r}. Set RAW_CSV_PATH in this cell to your local UserBehavior.csv."
    )
RAW_CSV_PATH = str(_Path(RAW_CSV_PATH).expanduser())
print("raw CSV:", RAW_CSV_PATH)


FileNotFoundError: raw CSV not found: 'data/UserBehavior.csv'. Set RAW_CSV_PATH in this cell to your local UserBehavior.csv.

## 1.1 · Data source and provenance

The raw `UserBehavior.csv` is the public Tianchi 649 / Kaggle-mirror dataset. Provenance (download URL, sha256, independent corroboration) is recorded in `data/source_manifest.json`; the expected raw-data audit (row counts, behavior distribution, time window, duplicates) is in `results/data_audit.json`.


In [ ]:
# ---- data provenance + expected raw-data audit ----
provenance = json.loads((DATA / "source_manifest.json").read_text())
audit = json.loads((RES / "data_audit.json").read_text())

print("dataset  :", provenance["dataset"])
print("source   :", provenance["actual_source_catalog_url"])
print("raw      :", f"{audit['raw_rows']:,} rows |", f"{audit['distinct_users_raw']:,} users |",
      f"{audit['distinct_items_raw']:,} items")
print("behavior :", audit["behavior_raw_values"])
print("window   :", audit["timestamp"]["window_definition"],
      "| rows in window:", f"{audit['timestamp']['rows_in_window']:,}")
print("duplicates removed:", audit["duplicates_in_window"]["exact_duplicate_rows_extra_copies"])
print("expected clean:", f"{audit['clean']['rows']:,} rows |", audit["clean"]["behavior_counts"])


## 2 · Clean the raw CSV → events + user buckets

Filter to the Shanghai time window, map behavior strings to integer codes, collapse exact duplicates, and build the md5 user buckets. Produces `data/processed/events_clean.parquet` (100,095,182 rows) and `data/processed/user_buckets.parquet`. Skipped when the outputs already exist.


In [ ]:
# ---- step 2: clean raw CSV -> events_clean.parquet + user_buckets.parquet ----
START, END = 1511539200, 1512316800   # Asia/Shanghai 2017-11-25 .. 2017-12-04
BEHAVIOR_CODE = {"pv": 1, "fav": 2, "cart": 3, "buy": 4}
COLS = ["user_id", "item_id", "category_id", "behavior_type", "ts"]

if not (PROC / "events_clean.parquet").exists():
    chunks, offset = [], 0
    reader = pd.read_csv(RAW_CSV_PATH, header=None, names=COLS, chunksize=5_000_000,
                         dtype={"user_id": "int32", "item_id": "int32", "category_id": "int32",
                                "behavior_type": "str", "ts": "int64"})
    with tqdm(desc="clean raw CSV", unit="chunk") as pbar:
        for ch in reader:
            ch["row_no"] = np.arange(offset, offset + len(ch), dtype=np.int64)
            offset += len(ch)
            ch = ch[(ch.ts >= START) & (ch.ts < END)]
            chunks.append(ch)
            pbar.update(1)
    df = pd.concat(chunks, ignore_index=True); del chunks

    df["behavior"] = df.behavior_type.map(BEHAVIOR_CODE).astype(np.int8)
    assert df.behavior.notna().all(), "unrecognised behaviour values"
    n_dup = int(df.duplicated(subset=["user_id", "item_id", "category_id", "behavior", "ts"]).sum())
    df = df.drop_duplicates(subset=["user_id", "item_id", "category_id", "behavior", "ts"], keep="first")

    ev = df[["row_no", "user_id", "item_id", "category_id", "behavior", "ts"]].copy()
    ev.to_parquet(PROC / "events_clean.parquet", index=False)

    users = pd.DataFrame({"user_id": np.sort(df.user_id.unique())})
    users["bucket"] = users.user_id.map(
        lambda u: int(hashlib.md5(str(int(u)).encode("utf-8")).hexdigest(), 16) % 100).astype(np.int16)
    users.to_parquet(PROC / "user_buckets.parquet", index=False)

    log("cleaned", len(ev), "rows | duplicates removed:", n_dup, "| users:", len(users))
else:
    log("events_clean.parquet already present; skipping cleaning")


In [ ]:
# ---- verify the cleaned table matches the recorded audit ----
ev = pd.read_parquet(PROC / "events_clean.parquet")
exp = json.loads((RES / "data_audit.json").read_text())["clean"]
code_map = {"pv": 1, "fav": 2, "cart": 3, "buy": 4}
exp_counts = {code_map[k]: v for k, v in exp["behavior_counts"].items()}
assert len(ev) == exp["rows"], (len(ev), exp["rows"])
assert ev.user_id.nunique() == exp["users"]
assert ev.item_id.nunique() == exp["items"]
assert ev.behavior.value_counts().to_dict() == exp_counts
print("OK: cleaned events match the recorded audit ->", f"{len(ev):,} rows")


## 3 · Build episodes from cleaned events

- Input: `data/processed/events_clean.parquet` (produced in step 2 above).
- Collapse each user × item pair into one episode: `t0` = the pair's first pv in the study window.
- Eligibility: no fav/cart/buy of the pair before `t0`; ≥24 h visible history before `t0`; `t0 + 24h ≤` window end.
- Output: `data/episodes_hour.parquet` (51,316,512 episodes).

~6.5 min.


In [ ]:
# ---- step 1: build user-item first-view episodes ----
START, END, DAY, HOUR = 1511539200, 1512316800, 86400, 3600
PV, BUY = 1, 4

def window_count(sorted_key, base, lo, hi):
    """Count of sorted_key values in [base+lo, base+hi)."""
    return (np.searchsorted(sorted_key, base + hi, "left") - np.searchsorted(sorted_key, base + lo, "left")).astype(np.int32)

ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["user_id", "item_id", "behavior", "ts"])
u = ev.user_id.to_numpy().astype(np.int64); it = ev.item_id.to_numpy().astype(np.int64)
b = ev.behavior.to_numpy(); ts = ev.ts.to_numpy().astype(np.int64)
assert u.max() < 2 ** 20 and it.max() < 2 ** 23
key = (u << 23) | it
log("events", len(ev))

pv = b == PV
first_pv = pd.Series(ts[pv]).groupby(key[pv]).min()
nonpv = ~pv
first_nonpv = pd.Series(ts[nonpv]).groupby(key[nonpv]).min()
isbuy = b == BUY
first_buy = pd.Series(ts[isbuy]).groupby(key[isbuy]).min()
log("pairs with pv", len(first_pv))

k = first_pv.index.to_numpy(); t0 = first_pv.to_numpy()
fn = first_nonpv.reindex(k).to_numpy()
fb = first_buy.reindex(k).to_numpy()
wf = [("user_item_pairs_with_pv", len(k))]
m = t0 >= START + DAY; wf.append(("t0_after_1d_washout", int(m.sum())))
m &= t0 + DAY <= END; wf.append(("complete_24h_outcome_window", int(m.sum())))
m &= ~(fn < t0); wf.append(("no_fav_cart_buy_before_t0", int(m.sum())))
k, t0, fb = k[m], t0[m], fb[m]
assert not (fb < t0).any()

ep = pd.DataFrame({"user_id": (k >> 23).astype(np.int32), "item_id": (k & (2 ** 23 - 1)).astype(np.int32), "t0": t0})
ep["buy_delay_s"] = np.where(np.isnan(fb), -1, fb - t0).astype(np.int64)
ep["hour"] = (((t0 - START) % DAY) // HOUR).astype(np.int8)
ep["day"] = ((t0 - START) // DAY).astype(np.int8)

# user history (other items only: the focal pair has no event before t0 by construction)
uk_pv = np.sort((u[pv] << 32) + ts[pv]); uk_buy = np.sort((u[isbuy] << 32) + ts[isbuy])
ub = ep.user_id.to_numpy().astype(np.int64) << 32
ep["hist_pv24"] = window_count(uk_pv, ub, t0 - DAY, t0)
ep["hist_pv1h"] = window_count(uk_pv, ub, t0 - HOUR, t0)
ep["hist_buy24"] = (window_count(uk_buy, ub, t0 - DAY, t0) > 0).astype(np.int8)
del uk_pv, uk_buy
ik_pv = np.sort((it[pv] << 32) + ts[pv])
ep["pop24"] = window_count(ik_pv, ep.item_id.to_numpy().astype(np.int64) << 32, t0 - DAY, t0)
del ik_pv

bk = pd.read_parquet(PROC / "user_buckets.parquet").set_index("user_id").bucket
ep["bucket"] = bk.reindex(ep.user_id.to_numpy()).to_numpy().astype(np.int16)
cnt = pd.Series(u).value_counts()
thr = float(cnt.quantile(.99))
ep["top1pct_user"] = ep.user_id.isin(cnt.index[cnt > thr]).astype(np.int8)
ep.to_parquet(DATA / "episodes_hour.parquet", index=False)
summary = {"waterfall": wf, "episodes": len(ep), "users": int(ep.user_id.nunique()),
           "items": int(ep.item_id.nunique()), "top1pct_user_event_threshold": thr,
           "top1pct_users": int((cnt > thr).sum())}
(RES / "sample_waterfall.json").write_text(json.dumps(summary, indent=2))
log(summary)


## 3.1 · Episode waterfall, buyer timing and integrity checks

After building the episode table, confirm the funnel and that the table is clean.


In [ ]:
# ---- episode funnel + when buyers actually buy ----
wf = json.loads((RES / "sample_waterfall.json").read_text())
print("episode funnel:")
for name, n in wf["waterfall"]:
    print(f"  {name:35s} {n:,}")
print("episodes:", f"{wf['episodes']:,} | users:", f"{wf['users']:,} | items:", f"{wf['items']:,}")

ep = pd.read_parquet(DATA / "episodes_hour.parquet")
dl = ep.buy_delay_s.to_numpy()
buyers = dl[(dl >= 0) & (dl < 86400)]
print(f"share of 24h buyers who buy within 1h : {(buyers < 3600).mean()*100:.1f}%")
print(f"median delay of 24h buyers            : {np.median(buyers)/60:.1f} min")


In [ ]:
# ---- data integrity checks on the episode table ----
checks = {
    "missing values (any column)": int(ep[["user_id","item_id","t0","buy_delay_s","hour","day","pop24","hist_pv24","hist_pv1h","hist_buy24"]].isna().sum().sum()),
    "duplicate user-item episodes": int(ep.duplicated(["user_id","item_id"]).sum()),
    "hour outside 0..23": int((~ep.hour.between(0, 23)).sum()),
    "buy_delay_s < -1": int((ep.buy_delay_s < -1).sum()),
    "negative history counts": int((ep[["hist_pv24","hist_pv1h","pop24"]] < 0).sum().sum()),
}
print(pd.Series(checks, name="count"))
assert sum(checks.values()) == 0, checks
print("OK: episode table integrity checks all pass")


## 4 · Fixed-effects linear probability model (FE-LPM)

- Outcomes (pp, 0/100): Y1 = buy within 1 h, Y2 = within 24 h, Y10m = within 10 min, Y3 = deferred buy.
- Time bins: day 09–17 (ref), evening 18–21, late 22–01, small 02–05, morning 06–08.
- Model sequence: M0 (no FE) → M1 (+user FE) → **M2 main** (user + item + day FE + pre-t0 controls).
- SE: user × item two-way clustering; also user × dayhour (168 clusters) as sensitivity S4.
- 4 primary tests with Holm correction; 7 pre-specified robustness checks (S1–S7).

~27 min on the 9.4 M-episode main sample.


In [ ]:
# ---- step 2: FE LPM definitions ----
BINS = {"day": range(9, 18), "evening": range(18, 22), "late": [22, 23, 0, 1], "small": range(2, 6), "morning": range(6, 9)}
DUM = ["evening", "late", "small", "morning"]
CTRL = ["l_hpv24", "hist_buy24", "l_pop"]
HOURS = [f"h{h:02d}" for h in range(24) if h != 14]
FE_MAIN = ["user_id", "item_id", "day"]

def derive(ep):
    d = ep
    dl = d.buy_delay_s.to_numpy()
    bought = dl >= 0
    d["y1"] = np.where(bought & (dl < 3600), 100.0, 0.0)
    d["y2"] = np.where(bought & (dl < 86400), 100.0, 0.0)
    d["y10m"] = np.where(bought & (dl < 600), 100.0, 0.0)
    d["y3"] = np.where(bought & (dl >= 3600) & (dl < 86400), 100.0, 0.0)
    hb = pd.Series(index=range(24), dtype=object)
    for name, hs in BINS.items():
        hb[list(hs)] = name
    d["bin"] = hb.to_numpy()[d.hour.to_numpy()]
    for c in DUM:
        d[c] = (d.bin == c).astype(float)
    d["l_hpv24"] = np.log2(1 + d.hist_pv24); d["l_hpv1h"] = np.log2(1 + d.hist_pv1h)
    d["l_pop"] = np.log2(1 + d.pop24); d["hist_buy24"] = d.hist_buy24.astype(float)
    d["dayhour"] = d.day.astype(np.int32) * 24 + d.hour
    return d

def fit_multi(d, xcols, fes, ycols, name, clusters=(("user_id", "item_id"),)):
    """One residualization, OLS for every outcome and every clustering."""
    t = time.time()
    cols = xcols + ycols
    if fes:
        ids = np.column_stack([pd.factorize(d[c], sort=False)[0] for c in fes])
        opts = {"tol": 1e-9, "iteration_limit": 20000, "transform": "symmetric", "acceleration": "cg"} if len(fes) > 1 else None
        algo = pyhdfe.create(ids, drop_singletons=True, compute_degrees=True, degrees_method="pairwise" if len(fes) <= 2 else "simple",
                             residualize_method="map" if len(fes) > 1 else "within", options=opts)
        keep = ~algo.singleton_indices if algo.singleton_indices is not None else np.ones(len(d), bool)
        Z = algo.residualize(d[cols].to_numpy(float)); g = d.loc[keep]; deg = int(algo.degrees)
        X = Z[:, :len(xcols)]; Ys = Z[:, len(xcols):]; names = list(xcols)
    else:
        X = np.column_stack([np.ones(len(d)), d[xcols].to_numpy(float)]); Ys = d[ycols].to_numpy(float); g = d; deg = 0
        names = ["const"] + list(xcols)
    assert np.linalg.matrix_rank(X) == X.shape[1], name
    rows = []
    for j, yc in enumerate(ycols):
        y = Ys[:, j]
        beta = np.linalg.lstsq(X, y, rcond=None)[0]; e = y - X @ beta
        ref_mean = float(g.loc[g.bin == "day", yc].mean()) if "bin" in g else np.nan
        for cl in clusters:
            cov, counts, mineig = cluster_cov(X, e, [g[c].to_numpy() for c in cl], deg + X.shape[1])
            se = np.sqrt(np.diag(cov)); dfree = min(counts[:len(cl)]) - 1; tc = stats.t.ppf(.975, dfree)
            r = pd.DataFrame({"term": names, "coef": beta, "se": se, "ci_low": beta - tc * se, "ci_high": beta + tc * se,
                              "p": 2 * stats.t.sf(np.abs(beta / se), dfree)})
            r["outcome"] = yc; r["model"] = name; r["cluster"] = "+".join(cl); r["n"] = len(g)
            r["n_dropped_singletons"] = len(d) - len(g); r["users"] = g.user_id.nunique(); r["items"] = g.item_id.nunique()
            r["y_mean"] = float(g[yc].mean()); r["day_ref_mean"] = ref_mean; r["fe"] = "+".join(fes) if fes else "none"
            r["clusters_1"] = counts[0]; r["clusters_2"] = counts[1] if len(counts) > 1 else np.nan; r["min_cov_eig"] = mineig
            rows.append(r)
    log(name, "n", len(g), f"{time.time() - t:.0f}s")
    return pd.concat(rows)

def describe(full):
    out = []
    for g in ["hour", "bin"]:
        t = full.groupby(g).agg(episodes=("y1", "size"), users=("user_id", "nunique"), y1_pct=("y1", "mean"), y2_pct=("y2", "mean"),
                                y10m_pct=("y10m", "mean")).reset_index().rename(columns={g: "level"})
        nb = full[full.y1 == 0].groupby(g).y3.agg(["size", "mean"]).rename(columns={"size": "not_bought_1h", "mean": "y3_pct_given_no_1h"})
        t = t.join(nb, on="level")
        t["share_of_episodes_pct"] = t.episodes / t.episodes.sum() * 100
        t["share_of_24h_buys_pct"] = (t.y2_pct * t.episodes) / (t.y2_pct * t.episodes).sum() * 100
        t["grouping"] = g; out.append(t)
    return pd.concat(out)


In [ ]:
# ---- step 2: FE run ----
from statsmodels.stats.multitest import multipletests

full = derive(pd.read_parquet(DATA / "episodes_hour.parquet"))
describe(full).to_csv(RES / "descriptives_full.csv", index=False)
coefs = [fit_multi(full, DUM, [], ["y1", "y2"], "M0_full")]
main_s = full[full.bucket.between(40, 59)].copy(); rep_s = full[full.bucket.between(60, 79)].copy()
del full
summ = {s: {"episodes": len(d), "users": int(d.user_id.nunique())} for s, d in [("main", main_s), ("replication", rep_s)]}
coefs.append(fit_multi(main_s, DUM, [], ["y1", "y2"], "M0_main"))
coefs.append(fit_multi(main_s, DUM, ["user_id"], ["y1", "y2"], "M1_main"))
m2 = fit_multi(main_s, DUM + CTRL, FE_MAIN, ["y1", "y2", "y10m"], "M2_main",
               clusters=(("user_id", "item_id"), ("user_id", "dayhour")))
coefs.append(m2)
coefs.append(fit_multi(rep_s, DUM + CTRL, FE_MAIN, ["y1", "y2"], "M2_replication"))
coefs = pd.concat(coefs); coefs.to_csv(RES / "model_coefficients.csv", index=False)

base = m2[(m2.cluster == "user_id+item_id")]
prim = pd.concat([base[(base.term == t) & (base.outcome == y)] for t in ["late", "small"] for y in ["y1", "y2"]])
prim["p_holm"] = multipletests(prim.p, method="holm")[1]
prim["coef_over_day_ref_pct"] = prim.coef / prim.day_ref_mean * 100
prim.to_csv(RES / "primary_tests.csv", index=False)

rob = [base[base.outcome.isin(["y1", "y2"])].assign(robustness="main"),
       base[base.outcome == "y10m"].assign(robustness="S3_y1_10min"),
       m2[m2.cluster == "user_id+dayhour"].assign(robustness="S4_cluster_user_dayhour")]
rob.append(fit_multi(main_s, DUM, FE_MAIN, ["y1", "y2"], "S1_no_controls").assign(robustness="S1_no_controls"))
rob.append(fit_multi(main_s, DUM + CTRL + ["l_hpv1h"], FE_MAIN, ["y1", "y2"], "S2_session").assign(robustness="S2_plus_prior_1h_pv"))
rob.append(fit_multi(main_s[main_s.day >= 3].copy(), DUM + CTRL, FE_MAIN, ["y1", "y2"], "S5_washout72").assign(robustness="S5_washout72"))
rob.append(fit_multi(main_s[main_s.top1pct_user == 0].copy(), DUM + CTRL, FE_MAIN, ["y1", "y2"], "S6_excl_top1").assign(robustness="S6_excl_top1pct"))
rob.append(coefs[coefs.model == "M2_replication"].assign(robustness="S7_replication"))
pd.concat(rob).to_csv(RES / "robustness.csv", index=False)

nb = main_s[main_s.y1 == 0].copy()
fit_multi(nb, DUM + CTRL, FE_MAIN, ["y3"], "Y3_deferred_given_no_1h").to_csv(RES / "secondary_y3.csv", index=False)

for h in range(24):
    if h != 14:
        main_s[f"h{h:02d}"] = (main_s.hour == h).astype(float)
fit_multi(main_s, HOURS + CTRL, FE_MAIN, ["y1", "y2"], "M2_hour_shape").to_csv(RES / "hour_shape_M2.csv", index=False)
(RES / "model_summary.json").write_text(json.dumps(summ, indent=2))
log("done")


## 4.1 · Reproducibility check

The fixed-effects step refits everything from scratch. Confirm the freshly-fitted primary tests match the saved results.


In [ ]:
# ---- fresh FE coefficients vs saved results ----
saved = pd.read_csv(RES / "primary_tests.csv").set_index(["term", "outcome"])
fresh = prim.set_index(["term", "outcome"])
delta = float((fresh.coef - saved.coef).abs().max())
print("max |fresh - saved| on primary tests:", delta)
assert delta < 1e-9
print("OK: FE primary tests reproduce the saved results")


## 5 · Independent verification

(1) rebuild episodes for 400 random users with row-wise pandas and compare field-by-field; (2) local timezone anchor check; (3) absorbed-FE estimator vs explicit-dummy OLS on a small subsample.


In [ ]:
# ---- step 3: independent checks ----
from datetime import datetime, timezone, timedelta
import statsmodels.api as sm

START, END, DAY = 1511539200, 1512316800, 86400
SH = timezone(timedelta(hours=8))

def rebuild(ev_u):
    rows = []
    for (uid, iid), g in ev_u.groupby(["user_id", "item_id"]):
        pv = g[g.behavior == 1]
        if pv.empty:
            continue
        t0 = int(pv.ts.min())
        if t0 < START + DAY or t0 + DAY > END:
            continue
        if (g[(g.behavior != 1)].ts < t0).any():
            continue
        buys = g[(g.behavior == 4) & (g.ts >= t0)].ts
        lt = datetime.fromtimestamp(t0, SH)
        uall = ev_u[ev_u.user_id == uid]
        rows.append({"user_id": uid, "item_id": iid, "t0": t0, "buy_delay_s": int(buys.min() - t0) if len(buys) else -1,
                     "hour": lt.hour, "day": (lt.date() - datetime(2017, 11, 25).date()).days,
                     "hist_pv24": int(((uall.behavior == 1) & (uall.ts >= t0 - DAY) & (uall.ts < t0)).sum()),
                     "hist_pv1h": int(((uall.behavior == 1) & (uall.ts >= t0 - 3600) & (uall.ts < t0)).sum()),
                     "hist_buy24": int(((uall.behavior == 4) & (uall.ts >= t0 - DAY) & (uall.ts < t0)).any())})
    return pd.DataFrame(rows)

out = {}
assert datetime.fromtimestamp(START, SH).isoformat() == "2017-11-25T00:00:00+08:00"
out["start_anchor"] = datetime.fromtimestamp(START, SH).isoformat()
ep = pd.read_parquet(DATA / "episodes_hour.parquet")
rng = np.random.default_rng(5312)
users = rng.choice(ep.user_id.unique(), 400, replace=False)
ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["user_id", "item_id", "behavior", "ts"],
                     filters=[("user_id", "in", users.tolist())])
ref = rebuild(ev)
pipe = ep[ep.user_id.isin(users)]
cols = ["buy_delay_s", "hour", "day", "hist_pv24", "hist_pv1h", "hist_buy24"]
mg = pipe.merge(ref, on=["user_id", "item_id"], how="outer", suffixes=("_p", "_r"), indicator=True)
out["rebuild"] = {"users": 400, "pipeline_episodes": len(pipe), "reference_episodes": len(ref),
                  "only_one_side": int((mg._merge != "both").sum()),
                  "field_mismatches": {c: int((mg[f"{c}_p"] != mg[f"{c}_r"]).sum()) for c in cols + ["t0"]},
                  "buys_within_24h": int(((ref.buy_delay_s >= 0) & (ref.buy_delay_s < DAY)).sum())}
# pop24 independent recount for 300 random episodes
smp = pipe.sample(300, random_state=1)
ev_i = pd.read_parquet(PROC / "events_clean.parquet", columns=["item_id", "behavior", "ts"],
                       filters=[("item_id", "in", smp.item_id.unique().tolist()), ("behavior", "==", 1)])
pop = [int(((ev_i.item_id == r.item_id) & (ev_i.ts >= r.t0 - DAY) & (ev_i.ts < r.t0)).sum()) for r in smp.itertuples()]
out["pop24_mismatches_of_300"] = int((np.array(pop) != smp.pop24.to_numpy()).sum())

# FE estimator vs explicit dummies (user + day FE, small subsample, single cluster set)
d = derive(ep[ep.bucket == 40].copy())
d = d[d.user_id.isin(d.user_id.drop_duplicates().sample(1500, random_state=2))].copy()
r = fit_multi(d, DUM + CTRL, ["user_id", "day"], ["y1"], "check", clusters=(("user_id",),))
dd = d.loc[d.groupby("user_id").y1.transform("size") > 1]
X = pd.concat([dd[DUM + CTRL], pd.get_dummies(dd.user_id, prefix="u", dtype=float),
               pd.get_dummies(dd.day, prefix="d", drop_first=True, dtype=float)], axis=1)
ols = sm.OLS(dd.y1.to_numpy(), X.to_numpy()).fit(cov_type="cluster", cov_kwds={"groups": pd.factorize(dd.user_id)[0]})
b = pd.Series(ols.params[:len(DUM + CTRL)], index=DUM + CTRL)
s = pd.Series(ols.bse[:len(DUM + CTRL)], index=DUM + CTRL)
rr = r.set_index("term")
out["fe_vs_dummies"] = {"n": len(dd), "max_abs_coef_diff": float((rr.coef - b).abs().max()),
                        "se_ratio_min": float((rr.se / s).min()), "se_ratio_max": float((rr.se / s).max())}
(RES / "verification.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))


## 6 · Pre-t0 features + independent recount

Add cumulative user pv/fav/cart/buy counts before `t0`, and the item's buys by other users in `[t0-24h, t0)`. Then recount 300 episodes independently and verify the train/val/test user overlap is 0.

~2 min.


In [ ]:
# ---- step 4a: pre-t0 features ----
START, DAY = 1511539200, 86400

def count_in(sorted_key, base, lo, hi):
    return (np.searchsorted(sorted_key, base + hi, "left") - np.searchsorted(sorted_key, base + lo, "left")).astype(np.int32)

ep = pd.read_parquet(DATA / "episodes_hour.parquet")
ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["user_id", "item_id", "behavior", "ts"])
log("loaded", len(ep), "episodes", len(ev), "events")
t0 = ep.t0.to_numpy().astype(np.int64)
ub = ep.user_id.to_numpy().astype(np.int64) << 32
u = ev.user_id.to_numpy().astype(np.int64); b = ev.behavior.to_numpy(); ts = ev.ts.to_numpy().astype(np.int64)
for code_, name in [(1, "u_pv_cum"), (2, "u_fav_cum"), (3, "u_cart_cum"), (4, "u_buy_cum")]:
    m = b == code_
    key = np.sort((u[m] << 32) + ts[m])
    ep[name] = count_in(key, ub, np.int64(START), t0)
    log(name, "mean", float(ep[name].mean()))
    del key
m = b == 4
ikey = np.sort((ev.item_id.to_numpy()[m].astype(np.int64) << 32) + ts[m])
ep["i_buy24"] = count_in(ikey, ep.item_id.to_numpy().astype(np.int64) << 32, t0 - DAY, t0)
log("i_buy24 mean", float(ep.i_buy24.mean()))
assert (ep.u_pv_cum >= ep.hist_pv24).all()
ep.to_parquet(DATA / "episodes_features.parquet", index=False)
(RES / "features_summary.json").write_text(json.dumps(
    {c: {"mean": float(ep[c].mean()), "p99": float(ep[c].quantile(.99))} for c in ["u_pv_cum", "u_fav_cum", "u_cart_cum", "u_buy_cum", "i_buy24"]}, indent=2))
log("saved", len(ep))


In [ ]:
# ---- step 4b: independent recount of features + split overlap check ----
ep = pd.read_parquet(DATA / "episodes_features.parquet")
s = ep.sample(300, random_state=11)
ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["user_id", "item_id", "behavior", "ts"],
                     filters=[("user_id", "in", s.user_id.unique().tolist())])
evi = pd.read_parquet(PROC / "events_clean.parquet", columns=["item_id", "behavior", "ts"],
                      filters=[("item_id", "in", s.item_id.unique().tolist()), ("behavior", "==", 4)])
mism = {c: 0 for c in ["u_pv_cum", "u_fav_cum", "u_cart_cum", "u_buy_cum", "i_buy24"]}
for r in s.itertuples():
    uu = ev[(ev.user_id == r.user_id) & (ev.ts >= START) & (ev.ts < r.t0)]
    for code_, c in [(1, "u_pv_cum"), (2, "u_fav_cum"), (3, "u_cart_cum"), (4, "u_buy_cum")]:
        mism[c] += int((uu.behavior == code_).sum() != getattr(r, c))
    mism["i_buy24"] += int(((evi.item_id == r.item_id) & (evi.ts >= r.t0 - DAY) & (evi.ts < r.t0)).sum() != r.i_buy24)
users = {k: set(ep.user_id[ep.bucket.between(lo, hi)]) for k, (lo, hi) in [("train", (0, 59)), ("val", (60, 79)), ("test", (80, 99))]}
out = {"feature_mismatches_of_300": mism,
       "split_user_overlap": {f"{a}&{b}": len(users[a] & users[b]) for a, b in [("train", "val"), ("train", "test"), ("val", "test")]},
       "split_sizes": {k: int(ep.bucket.between(lo, hi).sum()) for k, (lo, hi) in [("train", (0, 59)), ("val", (60, 79)), ("test", (80, 99))]},
       "split_users": {k: len(v) for k, v in users.items()}}
(RES / "features_verification.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))


## 7 · Survival analysis + FastCox verification

- Event: first buy in `[t0, t0+24h)`; buys at ≥24 h censored at 24 h (time in hours).
- Cox (FastCox, Breslow ties, user-clustered Lin-Wei SE): M1 (bins) → M2 (+controls+day) → M3 (stratified by user×item decile×day).
- Piecewise Cox: `[0,1h)` and `[1h,24h)` (among those still at risk at 1 h).
- AFT Weibull (lifelines, naive SE, direction only); C-index on 500 k random test episodes.

> **Heavy**: `TAG='smoke'` for a quick check; `TAG='full'` (~12 min / 17 GB). Saved full results are in `results/`.


In [ ]:
# ---- step 5a: survival analysis ----
from lifelines import WeibullAFTFitter
from lifelines.utils import concordance_index

DUM = ["evening", "late", "small", "morning"]
CTRL = ["l_hpv24", "hist_buy24", "l_pop24"]
DAYD = [f"d{k}" for k in range(2, 8)]  # ref day 1; day 8 merged into day 7
CHECKPOINTS_H = [1 / 6, 1, 3, 6, 12, 24]

def prep(d):
    dl = d.buy_delay_s.to_numpy()
    ev = (dl >= 0) & (dl < 86400)
    d["event"] = ev.astype(np.int8)
    d["dur_h"] = np.where(ev, dl / 3600.0, 24.0)
    for c in DUM:
        d[c] = (d.bin == c).astype(np.float64)
    dd = np.minimum(d.day.to_numpy(), 7)
    for k in range(2, 8):
        d[f"d{k}"] = (dd == k).astype(np.float64)
    return d

def cox(d, cols, dur, event, strata=None, name=""):
    t = time.time()
    f = FastCox(d[dur].to_numpy(), d[event].to_numpy(), d[cols].to_numpy(np.float64),
                None if strata is None else d[strata].to_numpy()).fit()
    f.robust_cov(d.user_id.to_numpy())
    r = f.summary(cols)
    r["model"] = name; r["n"] = len(d); r["events"] = int(d[event].sum()); r["clusters"] = f.n_clusters
    r["n_iter"] = f.n_iter; r["loglik"] = f.loglik
    log(name, "n", len(d), "events", int(d[event].sum()), "iter", f.n_iter, f"{time.time() - t:.0f}s",
        "late HR", round(float(r.loc[r.term == "late", "hr"].iloc[0]), 4))
    return r, f

def cindex(beta, d, cols, n=500_000):
    s = d.sample(min(n, len(d)), random_state=5312)
    return float(concordance_index(s.dur_h, -(s[cols].to_numpy(np.float64) @ beta), s.event))

def describe_surv(d):
    out = []
    dl = d.buy_delay_s.to_numpy()
    for b in BINS:
        m = (d.bin == b).to_numpy()
        row = {"bin": b, "episodes": int(m.sum()), "users": int(d.user_id[m].nunique())}
        for h in CHECKPOINTS_H:
            row[f"cum_buy_pct_{h:g}h"] = float(((dl[m] >= 0) & (dl[m] < h * 3600)).mean() * 100)
        buyers = dl[m][(dl[m] >= 0) & (dl[m] < 86400)]
        row["buyers_24h"] = int(len(buyers))
        row["median_delay_min_buyers24h"] = float(np.median(buyers) / 60)
        row["share_of_24h_buys_within_1h_pct"] = float((buyers < 3600).mean() * 100)
        out.append(row)
    return pd.DataFrame(out)

a = _Args(train="0-59", test="80-99", tag=TAG, aft_user_frac=1.0)
out_dir = RES if a.tag == "full" else RES / "smoke"; out_dir.mkdir(exist_ok=True)
info = {"args": vars(a), "timing_s": {}}
t0 = time.time()
tr = prep(load([parse_range(a.train)]))
tr["user_decile"] = pd.qcut(tr.groupby("user_id").user_id.transform("size").rank(method="first"), 10, labels=False)
tr["item_decile"] = pd.qcut(tr.groupby("item_id").item_id.transform("size").rank(method="first"), 10, labels=False)
tr["stratum"] = tr.user_decile * 1000 + tr.item_decile * 10 + np.minimum(tr.day, 7)
log("train", len(tr), "events", int(tr.event.sum()), "strata", tr.stratum.nunique())
M1, M2 = DUM, DUM + CTRL + DAYD
M3 = DUM + CTRL  # day is part of the stratum, so day dummies are not identified within strata
rows, betas = [], {}
for name, cols, strata in [("M1_bins", M1, None), ("M2_controls", M2, None), ("M3_stratified", M3, "stratum")]:
    r, f = cox(tr, cols, "dur_h", "event", strata, name); rows.append(r); betas[name] = (f.beta, cols)
p1 = tr[["user_id", "dur_h", "event"] + M2].copy()
p1["ev_0_1h"] = ((p1.event == 1) & (p1.dur_h < 1)).astype(np.int8); p1["dur_0_1h"] = np.minimum(p1.dur_h, 1.0)
r, _ = cox(p1, M2, "dur_0_1h", "ev_0_1h", None, "PW_0_1h"); rows.append(r)
p2 = p1[p1.dur_h >= 1].copy(); p2["dur_1h_24h"] = p2.dur_h - 1.0
r, _ = cox(p2, M2, "dur_1h_24h", "event", None, "PW_1h_24h"); rows.append(r)
del p1, p2
info["timing_s"]["cox"] = time.time() - t0
t1 = time.time()
aft_d = tr
if a.aft_user_frac < 1.0:
    us = tr.user_id.drop_duplicates().sample(frac=a.aft_user_frac, random_state=5312)
    aft_d = tr[tr.user_id.isin(us)]
aft = WeibullAFTFitter()
aft.fit(aft_d[["dur_h", "event"] + M2], duration_col="dur_h", event_col="event")
s = aft.summary.reset_index()
s = s[s.param == "lambda_"][["covariate", "coef", "se(coef)", "p"]].rename(columns={"covariate": "term", "se(coef)": "se"})
s["time_ratio"] = np.exp(s.coef); s["model"] = "AFT_weibull_naive_se"; s["n"] = len(aft_d); s["events"] = int(aft_d.event.sum())
rows.append(s); info["aft_rho"] = float(np.exp(aft.params_.loc[("rho_", "Intercept")]))
info["timing_s"]["aft"] = time.time() - t1; log("aft", round(info["timing_s"]["aft"]), "s")
del tr, aft_d
te = prep(load([parse_range(a.test)]))
describe_surv(te).to_csv(out_dir / f"survival_cumulative_by_bin_{a.tag}.csv", index=False)
info["c_index_test"] = {k: cindex(b, te, c) for k, (b, c) in betas.items() if k != "M3_stratified"}
info["n_test"] = len(te); info["test_events"] = int(te.event.sum())
pd.concat(rows).to_csv(out_dir / f"survival_models_{a.tag}.csv", index=False)
info["timing_s"]["total"] = time.time() - t0
(out_dir / f"survival_{a.tag}.json").write_text(json.dumps(info, indent=2))
log("done", round(info["timing_s"]["total"]), "s", info["c_index_test"])


In [ ]:
# ---- step 5b: verify FastCox vs statsmodels / lifelines ----
from statsmodels.duration.hazard_regression import PHReg
from lifelines import CoxPHFitter

d = prep(load([(0, 0)]))
cols = DUM + CTRL + DAYD
x = d.sample(40000, random_state=3).reset_index(drop=True)
x["dur_h"] = np.round(x.dur_h, 2)  # force many ties so the Breslow tie handling is exercised
out = {"n": len(x), "events": int(x.event.sum()), "distinct_event_times": int(x.dur_h[x.event == 1].nunique())}
fc = FastCox(x.dur_h, x.event, x[cols]).fit(); fc.robust_cov(x.user_id)
smph = PHReg(x.dur_h.to_numpy(), x[cols].to_numpy(float), status=x.event.to_numpy(), ties="breslow").fit(groups=x.user_id.to_numpy())
out["vs_statsmodels"] = {"max_abs_coef_diff": float(np.abs(fc.beta - smph.params).max()),
                         "loglik_diff": float(fc.loglik - smph.llf),
                         "robust_se_ratio_range": [float((np.sqrt(np.diag(fc.cov)) / smph.bse).min()), float((np.sqrt(np.diag(fc.cov)) / smph.bse).max())]}
xs = x.copy(); xs["st"] = xs.user_id % 3
fcs = FastCox(xs.dur_h, xs.event, xs[cols], strata=xs.st).fit()
sms = PHReg(xs.dur_h.to_numpy(), xs[cols].to_numpy(float), status=xs.event.to_numpy(), ties="breslow", strata=xs.st.to_numpy()).fit()
out["stratified_vs_statsmodels"] = {"max_abs_coef_diff": float(np.abs(fcs.beta - sms.params).max()),
                                    "loglik_diff": float(fcs.loglik - sms.llf)}
y = x.copy(); y["dur_h"] = y.dur_h + np.random.default_rng(1).uniform(0, 1e-4, len(y))  # no ties: Breslow = Efron
fu = FastCox(y.dur_h, y.event, y[cols]).fit(); fu.robust_cov(y.user_id)
lf = CoxPHFitter().fit(y[["dur_h", "event", "user_id"] + cols], "dur_h", "event", cluster_col="user_id")
out["no_ties_vs_lifelines_cluster"] = {"max_abs_coef_diff": float(np.abs(fu.beta - lf.params_[cols].to_numpy()).max()),
                                       "robust_se_ratio_range": [float((np.sqrt(np.diag(fu.cov)) / lf.standard_errors_[cols].to_numpy()).min()),
                                                                 float((np.sqrt(np.diag(fu.cov)) / lf.standard_errors_[cols].to_numpy()).max())]}
big = prep(load([(0, 4)]))
t = time.time(); FastCox(big.dur_h, big.event, big[cols]).fit().robust_cov(big.user_id)
out["timing_rows"] = len(big); out["timing_s"] = time.time() - t
(RES / "fastcox_verification.json").write_text(json.dumps(out, indent=2))
print(json.dumps(out, indent=2))


## 8 · Causal Forest (CausalForestDML)

- Sample: first views at late night (T=1) or day (T=0); outcome Y1×100 (pp); features are pre-t0 only.
- Split: train buckets 0–59 / val 60–79 / test 80–99 (user overlap 0).
- Nuisance: LightGBM with early stopping on val; forest 1,000 trees, honest, `groups=user_id` cross-fitting.
- Test evaluation: out-of-sample DML ATE, BLP, GATES quintiles, CLAN — all user-clustered.

> **Heavy**: `TAG='smoke'` for a quick check; `TAG='full'` (~51 min / 22 GB). `max_samples=0.05` required.


In [ ]:
# ---- step 6: Causal Forest ----
import lightgbm as lgb
from econml.dml import CausalForestDML

LGB = dict(learning_rate=0.05, num_leaves=63, min_child_samples=500, subsample=0.8, subsample_freq=1,
           colsample_bytree=0.9, n_jobs=8, verbose=-1, random_state=5312)

def early_stop_rounds(Xtr, ytr, Xva, yva, classifier, cap):
    M = lgb.LGBMClassifier if classifier else lgb.LGBMRegressor
    m = M(n_estimators=cap, **LGB)
    m.fit(Xtr, ytr, eval_set=[(Xva, yva)], callbacks=[lgb.early_stopping(50, verbose=False)])
    return int(m.best_iteration_ or cap)

def residual_reg(yt, tt, groups, extra=None):
    X = tt[:, None] if extra is None else np.column_stack([tt] + extra)
    b, cov, G = cluster_ols(yt, X, groups)
    se = np.sqrt(np.diag(cov))
    return b, se, cov, G

def evaluate(cf, d, Xcols, tag):
    """Out-of-sample residuals using the train-fitted nuisance models (fold-averaged)."""
    X = d[Xcols].to_numpy(np.float64)
    y = d.y1.to_numpy() * 100; t = d.treat.to_numpy().astype(np.float64)
    my = np.mean([m.predict(X) for m in cf.models_y[0]], axis=0)
    mt = np.mean([m.predict_proba(X)[:, 1] for m in cf.models_t[0]], axis=0)
    yt, tt = y - my, t - mt
    g = d.user_id.to_numpy()
    tau = cf.effect(X).ravel()
    out = {"sample": tag, "n": len(d), "users": int(d.user_id.nunique()), "treated_share": float(t.mean()),
           "y1_mean_pp": float(y.mean()), "propensity_range": [float(mt.min()), float(mt.max())]}
    b, se, _, G = residual_reg(yt, tt, g)
    out["dml_ate"] = {"coef": float(b[0]), "se": float(se[0]), "ci": [float(b[0] - 1.96 * se[0]), float(b[0] + 1.96 * se[0])],
                      "clusters": G}
    out["cate_pred"] = {"mean": float(tau.mean()), "sd": float(tau.std()),
                        "pct": {str(p): float(np.percentile(tau, p)) for p in [1, 5, 10, 25, 50, 75, 90, 95, 99]},
                        "share_negative": float((tau < 0).mean())}
    b, se, _, _ = residual_reg(yt, tt, g, [tt * (tau - tau.mean())])
    out["blp"] = {"alpha": float(b[0]), "alpha_se": float(se[0]), "beta": float(b[1]), "beta_se": float(se[1]),
                  "beta_ci": [float(b[1] - 1.96 * se[1]), float(b[1] + 1.96 * se[1])]}
    q = pd.qcut(tau, 5, labels=False, duplicates="drop")
    D = np.column_stack([(q == k) * tt for k in range(5)])
    b, se, cov, _ = residual_reg(yt, D[:, 0], g, [D[:, k] for k in range(1, 5)])
    gates = []
    for k in range(5):
        m = q == k
        row = {"quintile": k + 1, "n": int(m.sum()), "coef": float(b[k]), "se": float(se[k]),
               "ci": [float(b[k] - 1.96 * se[k]), float(b[k] + 1.96 * se[k])], "tau_hat_mean": float(tau[m].mean()),
               "raw_y1_late_pp": float(y[m & (t == 1)].mean()), "raw_y1_day_pp": float(y[m & (t == 0)].mean())}
        for c in Xcols:
            row[f"mean_{c}"] = float(d[c].to_numpy()[m].mean())
        gates.append(row)
    dlt = b[4] - b[0]; dse = float(np.sqrt(cov[4, 4] + cov[0, 0] - 2 * cov[0, 4]))
    out["gates"] = gates
    out["gates_q5_minus_q1"] = {"diff": float(dlt), "se": dse, "ci": [float(dlt - 1.96 * dse), float(dlt + 1.96 * dse)]}
    return out, tau

a = _Args(train="0-59", val="60-79", test="80-99", n_trees=1000, cap=2000, tag=TAG, skip_test=False, max_samples=0.05)
tr, va, te = parse_range(a.train), parse_range(a.val), parse_range(a.test)
out_dir = RES if a.tag == "full" else RES / "smoke"; out_dir.mkdir(exist_ok=True)
info = {"args": vars(a), "features": CF_FEATURES, "lgb": LGB, "timing_s": {}}
hours = LATE + DAYH
t0 = time.time()
d_tr = load([tr], hours); d_va = load([va], hours)
for d in (d_tr, d_va):
    d["treat"] = d.hour.isin(LATE).astype(np.int64)
log("train", len(d_tr), "val", len(d_va))
Xtr = d_tr[CF_FEATURES].to_numpy(np.float64); Xva = d_va[CF_FEATURES].to_numpy(np.float64)
ny = early_stop_rounds(Xtr, d_tr.y1.to_numpy() * 100, Xva, d_va.y1.to_numpy() * 100, False, a.cap)
nt = early_stop_rounds(Xtr, d_tr.treat.to_numpy(), Xva, d_va.treat.to_numpy(), True, a.cap)
info["nuisance_rounds"] = {"y": ny, "t": nt}; info["timing_s"]["early_stop"] = time.time() - t0
log("rounds y", ny, "t", nt)
cf = CausalForestDML(model_y=lgb.LGBMRegressor(n_estimators=ny, **LGB),
                     model_t=lgb.LGBMClassifier(n_estimators=nt, **LGB),
                     discrete_treatment=True, cv=2, n_estimators=a.n_trees, min_samples_leaf=500,
                     max_samples=a.max_samples, honest=True, inference=True, n_jobs=-1, random_state=5312)
t1 = time.time()
cf.fit(d_tr.y1.to_numpy() * 100, d_tr.treat.to_numpy(), X=Xtr, groups=d_tr.user_id.to_numpy())
info["timing_s"]["cf_fit"] = time.time() - t1; log("cf fit", round(info["timing_s"]["cf_fit"]), "s")
info["n_train"] = len(d_tr); info["train_users"] = int(d_tr.user_id.nunique())
info["feature_importances"] = dict(zip(CF_FEATURES, map(float, cf.feature_importances_)))
del d_tr, Xtr
ev, _ = evaluate(cf, d_va, CF_FEATURES, "val"); info["val"] = ev
log("val ate", ev["dml_ate"], "blp beta", ev["blp"]["beta"])
del d_va, Xva
if not a.skip_test:
    d_te = load([te], hours); d_te["treat"] = d_te.hour.isin(LATE).astype(np.int64)
    ev, tau = evaluate(cf, d_te, CF_FEATURES, "test"); info["test"] = ev
    pd.DataFrame({"quantile": np.linspace(0, 1, 1001), "cate_pp": np.quantile(tau, np.linspace(0, 1, 1001))}) \
        .to_csv(out_dir / f"cate_quantiles_{a.tag}.csv", index=False)
    log("test ate", ev["dml_ate"], "blp beta", ev["blp"]["beta"], "q5-q1", ev["gates_q5_minus_q1"])
info["timing_s"]["total"] = time.time() - t0
(out_dir / f"causal_forest_{a.tag}.json").write_text(json.dumps(info, indent=2))
log("done", round(info["timing_s"]["total"]), "s")


## 9 · Phase 2 A: survival by hour-of-day × three purchase windows

24 hourly groups (ref hour 14) × purchase windows 1h / 3h / 24h. Event = buy within the window; later buys censored at window end. Output `results/survival_hourly_*.csv`.

> `TAG='smoke'` for a quick check; `TAG='full'` (~8 min).


In [ ]:
# ---- step 7: PHASE2 A, one Cox per hour-of-day x purchase window ----
CTRL = ["l_hpv24", "hist_buy24", "l_pop24"]
DAYD = [f"d{k}" for k in range(2, 8)]
WINDOWS_H = [1, 3, 24]
REF_HOUR = 14
HOURS = [f"h{h:02d}" for h in range(24) if h != REF_HOUR]

def prep(d, window_h):
    """Event = buy within window_h hours; censored at window_h otherwise. Hourly dummies."""
    w_s = int(window_h * 3600)
    dl = d.buy_delay_s.to_numpy()
    ev = (dl >= 0) & (dl < w_s)
    d["event"] = ev.astype(np.int8)
    d["dur_h"] = np.where(ev, dl / 3600.0, float(window_h))
    for h in range(24):
        if h != REF_HOUR:
            d[f"h{h:02d}"] = (d.hour == h).astype(np.float64)
    dd = np.minimum(d.day.to_numpy(), 7)
    for k in range(2, 8):
        d[f"d{k}"] = (dd == k).astype(np.float64)
    return d

def cox_hour(d, cols, name=""):
    t = time.time()
    f = FastCox(d.dur_h.to_numpy(), d.event.to_numpy(), d[cols].to_numpy(np.float64)).fit()
    f.robust_cov(d.user_id.to_numpy())
    r = f.summary(cols)
    r["model"] = name; r["n"] = len(d); r["events"] = int(d.event.sum()); r["clusters"] = f.n_clusters
    r["n_iter"] = f.n_iter; r["loglik"] = f.loglik
    log(name, "n", len(d), "events", int(d.event.sum()), f"{time.time() - t:.0f}s")
    return r

def describe_hour(d):
    """Cumulative purchase rates by hour of day (raw, test set)."""
    out = []
    dl = d.buy_delay_s.to_numpy()
    for h in range(24):
        m = (d.hour == h).to_numpy()
        row = {"hour": h, "episodes": int(m.sum()), "users": int(d.user_id[m].nunique())}
        for w in [1 / 6, 1, 3, 6, 12, 24]:
            row[f"cum_buy_pct_{w:g}h"] = float(((dl[m] >= 0) & (dl[m] < w * 3600)).mean() * 100)
        out.append(row)
    return pd.DataFrame(out)

a = _Args(train="0-59", test="80-99", tag=TAG)
out_dir = RES if a.tag == "full" else RES / "smoke"; out_dir.mkdir(exist_ok=True)
info = {"args": vars(a), "timing_s": {}}
t0 = time.time()
rows = []
for w in tqdm(WINDOWS_H, desc="survival window"):
    tr = prep(load([parse_range(a.train)]), w)
    for name, cols in [("M1_hour", HOURS), ("M2_hour", HOURS + CTRL + DAYD)]:
        rows.append(cox_hour(tr, cols, f"{name}_w{w}h"))
    del tr
info["timing_s"]["cox"] = time.time() - t0
te_raw = load([parse_range(a.test)])
describe_hour(te_raw).to_csv(out_dir / f"survival_hourly_cumulative_{a.tag}.csv", index=False)
info["n_test"] = len(te_raw)
pd.concat(rows).to_csv(out_dir / f"survival_hourly_models_{a.tag}.csv", index=False)
info["timing_s"]["total"] = time.time() - t0
(out_dir / f"survival_hourly_{a.tag}.json").write_text(json.dumps(info, indent=2))
log("done", round(info["timing_s"]["total"], 1), "s")


## 10 · Phase 2 B: cold vs hot items

- Popularity = `item_n` (first-view episode count per item). Deciles defined on `item_n≥2`; `item_n=1` reported separately as decile −1 (no within-item comparison).
- Descriptive: raw day vs late conversion by decile.
- Adjusted: the same FE-LPM as M2, fit within each decile 0..9.

~3 min.


In [ ]:
# ---- step 8: PHASE2 B, cold vs hot item night loss ----
d = derive(pd.read_parquet(DATA / "episodes_hour.parquet"))
d = d[d.bucket.between(40, 59)].copy()
d["item_n"] = d.groupby("item_id").item_id.transform("size")
d["decile"] = -1
mask = d.item_n >= 2
d.loc[mask, "decile"] = pd.qcut(d.loc[mask, "item_n"].rank(method="first"), 10, labels=False)
d["decile"] = d.decile.astype(int)
n1 = int((d.item_n == 1).sum())
log("n", len(d), "items", d.item_id.nunique(), "users", d.user_id.nunique(),
    "n1_rows", n1, f"({n1 / len(d) * 100:.1f}%)")

# descriptive: raw day vs late conversion by decile (incl. decile -1)
day, late = d[d.bin == "day"], d[d.bin == "late"]
desc_rows = []
for i in list(range(-1, 10)):
    ds, dys, lts = d[d.decile == i], day[day.decile == i], late[late.decile == i]
    desc_rows.append({
        "decile": int(i),
        "n_episodes": int(len(ds)), "n_items": int(ds.item_id.nunique()), "n_users": int(ds.user_id.nunique()),
        "item_n_min": int(ds.item_n.min()) if len(ds) else np.nan,
        "item_n_max": int(ds.item_n.max()) if len(ds) else np.nan,
        "mean_pop24": float(ds.pop24.mean()),
        "day_y1_pct": float(dys.y1.mean()), "day_y2_pct": float(dys.y2.mean()),
        "late_y1_pct": float(lts.y1.mean()), "late_y2_pct": float(lts.y2.mean()),
        "late_minus_day_y1": float(lts.y1.mean() - dys.y1.mean()),
        "late_minus_day_y2": float(lts.y2.mean() - dys.y2.mean()),
    })
desc = pd.DataFrame(desc_rows)
desc["relative_late_day_y1_pct"] = desc.late_minus_day_y1 / desc.day_y1_pct * 100
desc["relative_late_day_y2_pct"] = desc.late_minus_day_y2 / desc.day_y2_pct * 100
desc.to_csv(RES / "cold_hot_items_descriptive.csv", index=False)
log("descriptive saved")

# adjusted: FE LPM (same as main M2) within each decile 0..9
fe_rows = []
for i in tqdm(range(10), desc="FE per decile"):
    dsub = d[d.decile == i].copy()
    r = fit_multi(dsub, DUM + CTRL, FE_MAIN, ["y1", "y2"], f"M2_decile_{i}")
    r["decile"] = i
    fe_rows.append(r)
fe = pd.concat(fe_rows)
fe.to_csv(RES / "cold_hot_items_fe.csv", index=False)
late_fe = fe[(fe.term == "late") & (fe.cluster == "user_id+item_id")].copy()
late_fe["relative_pct"] = late_fe.coef / late_fe.day_ref_mean * 100
late_fe[["decile", "outcome", "coef", "se", "ci_low", "ci_high", "p", "n", "day_ref_mean",
         "relative_pct"]].to_csv(RES / "cold_hot_items_fe_late_summary.csv", index=False)
log("fe saved")
print(desc.round(3).to_string(index=False))


## 11 · Category extension: survival by category

For the top-20 categories (by train episodes) plus an "other" bucket, re-run Cox with hour dummies (23, ref 14) and 5-bin dummies (evening/late/small/morning, ref day), each across three windows (1h/3h/24h).

> `TAG='smoke'` for a quick check; `TAG='full'` (~8 min).


In [ ]:
# ---- step 9: survival re-run grouped by product category ----
def item_category_map():
    """item_id -> most common category_id (anonymised)."""
    ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["item_id", "category_id"])
    m = ev.groupby("item_id")["category_id"].agg(lambda s: s.mode().iloc[0]).rename("category_id")
    return m

CTRL = ["l_hpv24", "hist_buy24", "l_pop24"]
DAYD = [f"d{k}" for k in range(2, 8)]
WINDOWS_H = [1, 3, 24]
REF_HOUR = 14
HOURS = [f"h{h:02d}" for h in range(24) if h != REF_HOUR]
BINS5 = ["evening", "late", "small", "morning"]
BIN_HOURS = {"evening": [18, 19, 20, 21], "late": [22, 23, 0, 1], "small": [2, 3, 4, 5], "morning": [6, 7, 8]}
TOP_N = 20

def prep_cat(d, window_h):
    w_s = int(window_h * 3600)
    dl = d.buy_delay_s.to_numpy()
    ev = (dl >= 0) & (dl < w_s)
    d["event"] = ev.astype(np.int8)
    d["dur_h"] = np.where(ev, dl / 3600.0, float(window_h))
    for h in range(24):
        if h != REF_HOUR:
            d[f"h{h:02d}"] = (d.hour == h).astype(np.float64)
    for b in BINS5:
        d[b] = d.hour.isin(BIN_HOURS[b]).astype(np.float64)
    dd = np.minimum(d.day.to_numpy(), 7)
    for k in range(2, 8):
        d[f"d{k}"] = (dd == k).astype(np.float64)
    return d

def cox_cat(d, cols, name):
    f = FastCox(d.dur_h.to_numpy(), d.event.to_numpy(), d[cols].to_numpy(np.float64)).fit()
    f.robust_cov(d.user_id.to_numpy())
    r = f.summary(cols)
    r["model"] = name; r["n"] = len(d); r["events"] = int(d.event.sum()); r["clusters"] = f.n_clusters
    return r

a = _Args(train="0-59", tag=TAG)
out_dir = RES if a.tag == "full" else RES / "smoke"
out_dir.mkdir(exist_ok=True)

log("load mapping + episodes")
catmap = item_category_map()
d = load([parse_range(a.train)])
d = d.join(catmap, on="item_id")
log("episodes", len(d), "items", d.item_id.nunique())

# top-N categories by episode count; rest -> OTHER
counts = d.groupby("category_id").size().sort_values(ascending=False)
top = list(counts.head(TOP_N).index)
d["group"] = np.where(d.category_id.isin(top), d.category_id.astype(str), "OTHER")
grp_order = [str(c) for c in top] + ["OTHER"]

rows, summ = [], []
for g in tqdm(grp_order, desc="category survival"):
    sub = d[d.group == g]
    cat_id = "OTHER" if g == "OTHER" else g
    n, users = len(sub), int(sub.user_id.nunique())
    day24 = float(((sub.buy_delay_s >= 0) & (sub.buy_delay_s < 86400) & (sub.hour.isin(range(9, 18)))).mean() * 100)
    late24 = float(((sub.buy_delay_s >= 0) & (sub.buy_delay_s < 86400) & (sub.hour.isin([22, 23, 0, 1]))).mean() * 100)
    row = {"group": g, "label": cat_id, "n": n, "users": users, "share_pct": round(n / len(d) * 100, 2),
           "day_24h_pct": round(day24, 3), "late_24h_pct": round(late24, 3)}
    for w in tqdm(WINDOWS_H, desc="survival window"):
        p = prep_cat(sub.copy(), w)
        for name, cols in [("M2_bins", BINS5 + CTRL + DAYD), ("M2_hour", HOURS + CTRL + DAYD)]:
            r = cox_cat(p, cols, f"{g}|{name}|w{w}h")
            r["group"] = g; r["cat_label"] = cat_id; r["window"] = w
            rows.append(r)
            if name == "M2_bins":
                for term in ["late", "small"]:
                    rr = r[r.term == term].iloc[0]
                    row[f"{term}_HR_w{w}h"] = round(float(rr.hr), 3)
                    row[f"{term}_HRlo_w{w}h"] = round(float(rr.hr_ci_low), 3)
                    row[f"{term}_HRhi_w{w}h"] = round(float(rr.hr_ci_high), 3)
        del p
    summ.append(row)
    log("group", cat_id, "n", n, f"late_HR_w24h={row.get('late_HR_w24h')}")

pd.concat(rows).to_csv(out_dir / f"category_survival_models_{a.tag}.csv", index=False)
pd.DataFrame(summ).to_csv(out_dir / f"category_survival_summary_{a.tag}.csv", index=False)
log("done", len(grp_order), "groups")


## 12 · Category extension: Causal Forest with category dummies

Categories (9,437 values) would explode if one-hot, so encode the top-15 categories (by train episodes) as binary dummies; the long tail is the implicit reference. Everything else mirrors step 6 exactly.

> `TAG='smoke'` for a quick check; `TAG='full'` (~60 min / 22 GB).


In [ ]:
# ---- step 10: Causal Forest with top-K category dummies ----
TOP_K = 15

def item_category_map():
    ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["item_id", "category_id"])
    return ev.groupby("item_id")["category_id"].agg(lambda s: s.mode().iloc[0]).rename("category_id")

a = _Args(train="0-59", val="60-79", test="80-99", n_trees=1000, cap=2000, tag=TAG, skip_test=False,
          max_samples=0.05, topk=TOP_K)
tr, va, te = parse_range(a.train), parse_range(a.val), parse_range(a.test)
out_dir = RES if a.tag == "full" else RES / "smoke"; out_dir.mkdir(exist_ok=True)

log("load + category map")
catmap = item_category_map()
hours = LATE + DAYH
d_tr = load([tr], hours).join(catmap, on="item_id")
d_va = load([va], hours).join(catmap, on="item_id")

# top-K categories by train episode count -> dummies
top = list(d_tr.groupby("category_id").size().sort_values(ascending=False).head(a.topk).index)
cat_feats = [f"cat_{int(c)}" for c in top]
for d in (d_tr, d_va):
    d["treat"] = d.hour.isin(LATE).astype(np.int64)
    for c, f in zip(top, cat_feats):
        d[f] = (d.category_id == c).astype(np.float64)
FEATURES = CF_FEATURES + cat_feats
log("features", len(FEATURES), "| top categories", top[:5], "...")

info = {"args": vars(a), "features": FEATURES, "cat_features": cat_feats,
        "top_category_ids": [int(c) for c in top], "lgb": LGB, "timing_s": {}}
t0 = time.time()
Xtr = d_tr[FEATURES].to_numpy(np.float64); Xva = d_va[FEATURES].to_numpy(np.float64)
ny = early_stop_rounds(Xtr, d_tr.y1.to_numpy() * 100, Xva, d_va.y1.to_numpy() * 100, False, a.cap)
nt = early_stop_rounds(Xtr, d_tr.treat.to_numpy(), Xva, d_va.treat.to_numpy(), True, a.cap)
info["nuisance_rounds"] = {"y": ny, "t": nt}; info["timing_s"]["early_stop"] = time.time() - t0
log("rounds y", ny, "t", nt)

cf = CausalForestDML(model_y=lgb.LGBMRegressor(n_estimators=ny, **LGB),
                     model_t=lgb.LGBMClassifier(n_estimators=nt, **LGB),
                     discrete_treatment=True, cv=2, n_estimators=a.n_trees, min_samples_leaf=500,
                     max_samples=a.max_samples, honest=True, inference=True, n_jobs=-1, random_state=5312)
t1 = time.time()
cf.fit(d_tr.y1.to_numpy() * 100, d_tr.treat.to_numpy(), X=Xtr, groups=d_tr.user_id.to_numpy())
info["timing_s"]["cf_fit"] = time.time() - t1; log("cf fit", round(info["timing_s"]["cf_fit"]), "s")
info["n_train"] = len(d_tr); info["train_users"] = int(d_tr.user_id.nunique())
imp = dict(zip(FEATURES, map(float, cf.feature_importances_)))
info["feature_importances"] = imp
info["category_importance_total"] = float(sum(imp[f] for f in cat_feats))
info["top_category_shares"] = {int(c): round(float((d_tr.category_id == c).mean() * 100), 2) for c in top}
del d_tr, Xtr

ev, _ = evaluate(cf, d_va, FEATURES, "val"); info["val"] = ev
log("val ate", ev["dml_ate"]["coef"], "blp beta", round(ev["blp"]["beta"], 3))
del d_va, Xva

if not a.skip_test:
    d_te = load([te], hours).join(catmap, on="item_id")
    d_te["treat"] = d_te.hour.isin(LATE).astype(np.int64)
    for c, f in zip(top, cat_feats):
        d_te[f] = (d_te.category_id == c).astype(np.float64)
    ev, tau = evaluate(cf, d_te, FEATURES, "test"); info["test"] = ev
    pd.DataFrame({"quantile": np.linspace(0, 1, 1001), "cate_pp": np.quantile(tau, np.linspace(0, 1, 1001))}) \
        .to_csv(out_dir / f"cate_quantiles_category_{a.tag}.csv", index=False)
    log("test ate", ev["dml_ate"]["coef"], "blp beta", round(ev["blp"]["beta"], 3), "q5-q1", round(ev["gates_q5_minus_q1"]["diff"], 3))
info["timing_s"]["total"] = time.time() - t0
(out_dir / f"causal_forest_category_{a.tag}.json").write_text(json.dumps(info, indent=2))
log("done", round(info["timing_s"]["total"], 1), "s")


## 13 · Figures 01–03 (fixed effects)

Fig 01 raw vs FE-adjusted hourly conversion; fig 02 primary tests (late/small vs day); fig 03 robustness.


In [ ]:
# ---- figures 01-03 ----
from matplotlib.font_manager import FontProperties
_CJK_PATH = "/Library/Fonts/Arial Unicode.ttf"
CJK = FontProperties(fname=_CJK_PATH)
apply()

def fig1_hourly_shape():
    desc = pd.read_csv(RES / "descriptives_full.csv")
    hr = desc[desc.grouping == "hour"].set_index("level")
    hour_shape = pd.read_csv(RES / "hour_shape_M2.csv")
    hs = hour_shape[(hour_shape.outcome == "y1") & (hour_shape.cluster == "user_id+item_id")].set_index("term")
    hs = hs.reindex([f"h{h:02d}" for h in range(24) if h != 14])
    hs["hour"] = [int(t[1:]) for t in hs.index]
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.plot(hr.index, hr.y1_pct, "o-", color="#7C3A47", lw=1.5)
    ax1.axvspan(22, 24, alpha=.12, color="#7C3A47")
    ax1.axvspan(0, 2, alpha=.12, color="#7C3A47")
    ax1.axvspan(2, 6, alpha=.22, color="#C98A95")
    ax1.axvspan(9, 18, alpha=.10, color="#B9ABAE")
    ax1.set_ylim(0.355, 0.90)
    ax1.text(1.0, 0.885, "late", ha="center", va="top", fontsize=7.5, color="#7C3A47", fontproperties=CJK)
    ax1.text(23.0, 0.885, "late", ha="center", va="top", fontsize=7.5, color="#7C3A47", fontproperties=CJK)
    ax1.text(4.0, 0.885, "small", ha="center", va="top", fontsize=7.5, color="#8E4A57", fontproperties=CJK)
    ax1.text(13.5, 0.368, "day (ref)", ha="center", va="bottom", fontsize=7.5, color="#7A6E6E", fontproperties=CJK)
    ax1.set_xlabel("Hour of day (Shanghai)"); ax1.set_ylabel("1-hour purchase rate (%)")
    ax1.set_title("(a) Raw conversion by hour"); ax1.grid(alpha=.2)
    ax2.axhline(0, color="gray", lw=.8, ls="--")
    ax2.errorbar(hs.hour, hs.coef, yerr=1.96 * hs.se, fmt="o", color="#7C3A47", capsize=3, markersize=4)
    ax2.axhline(0, color="k", lw=.5)
    ax2.set_xlabel("Hour (ref: 14)"); ax2.set_ylabel("M2 coef vs hour 14 (pp)")
    ax2.set_title("(b) User/item/day FE + controls"); ax2.grid(alpha=.2)
    plt.tight_layout(); plt.savefig(FIG / "01_hourly_shape.png", dpi=150, bbox_inches="tight")
    plt.savefig(FIG / "01_hourly_shape.svg", bbox_inches="tight"); plt.close()

def fig2_bin_comparison():
    prim = pd.read_csv(RES / "primary_tests.csv")
    bins = ["late", "small"]
    ys = ["y1", "y2"]
    fig, axes = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
    for j, y in enumerate(ys):
        ax = axes[j]
        sub = prim[prim.outcome == y]
        ypos = np.arange(len(bins))
        cols = ["#7C3A47", "#D8A7AF"]
        ax.barh(ypos, sub.coef, height=0.44, color=cols, zorder=3)
        ax.errorbar(sub.coef, ypos, xerr=1.96 * sub.se, fmt="none", ecolor="#403737", capsize=4, lw=0.9, zorder=4)
        for i, r in enumerate(sub.itertuples()):
            ax.text(r.coef / 2, i, f"{r.coef:.2f} pp", va="center", ha="center",
                    fontsize=9.5, color="white" if i == 0 else "#403737", fontweight="bold", zorder=5)
        ax.set_yticks(ypos); ax.set_yticklabels(["late (22-01)", "small (02-05)"])
        ax.axvline(0, color="#4A4545", lw=0.8)
        ax.set_xlim(min(sub.ci_low) - 0.05, 0.03)
        ax.set_xlabel("coef vs day (pp)")
        ax.set_title(f"({chr(97 + j)}) {'1-hour' if y == 'y1' else '24-hour'} purchase")
        ax.grid(axis="x", zorder=0)
    plt.tight_layout(); plt.savefig(FIG / "02_bin_comparison.png", dpi=150, bbox_inches="tight")
    plt.savefig(FIG / "02_bin_comparison.svg", bbox_inches="tight"); plt.close()

def fig3_robustness():
    rob = pd.read_csv(RES / "robustness.csv")
    keep = ["main", "S1_no_controls", "S2_plus_prior_1h_pv", "S5_washout72", "S6_excl_top1pct", "S7_replication"]
    label_map = {"main": "Main", "S1_no_controls": "No controls", "S2_plus_prior_1h_pv": "+ prior 1h pv",
                 "S5_washout72": "Washout 72h", "S6_excl_top1pct": "Excl top 1%", "S7_replication": "Replication"}
    r = rob[(rob.robustness.isin(keep)) & (rob.cluster == "user_id+item_id") & (rob.outcome.isin(["y1", "y2"])) & (rob.term == "late")].copy()
    # explicit display order so row i == keep[i]; do NOT sort alphabetically (labels are positional)
    r["pos"] = r.robustness.map({k: i for i, k in enumerate(keep)})
    r = r.sort_values(["pos", "outcome"])
    fig, ax = plt.subplots(figsize=(8, 4))
    for j, y in enumerate(["y1", "y2"]):
        sub = r[r.outcome == y]
        x = np.arange(len(sub)) + j * .35
        ax.barh(x, sub.coef, xerr=1.96 * sub.se, height=.3, label=f"{'1h' if y == 'y1' else '24h'} purchase",
                color="#7C3A47" if y == "y1" else "#A85D68", capsize=3, alpha=.85)
    ax.axvline(0, color="k", lw=.8)
    ax.set_yticks(np.arange(len(keep)) + .175)
    ax.set_yticklabels([label_map[k] for k in keep])
    ax.set_xlabel("late vs day coef (pp)"); ax.set_title("Robustness: late (22-01) vs day (09-17)")
    ax.legend(loc="lower right"); ax.grid(axis="x", alpha=.2)
    plt.tight_layout(); plt.savefig(FIG / "03_robustness.png", dpi=150, bbox_inches="tight")
    plt.savefig(FIG / "03_robustness.svg", bbox_inches="tight"); plt.close()

fig1_hourly_shape(); fig2_bin_comparison(); fig3_robustness()
print("figures 01-03 written")


## 14 · Figures 04–06 (survival + Causal Forest)

Fig 04 test-set cumulative purchase curves; fig 05 piecewise Cox HRs; fig 06 GATES + CATE distribution.


In [ ]:
# ---- figures 04-06 ----
COL = {"day": "#403737", "evening": "#A85D68", "late": "#7C3A47", "small": "#D8A7AF", "morning": "#C98A95"}
LAB = {"day": "Day 09-17 (ref)", "evening": "Evening 18-21", "late": "Late 22-01", "small": "Small hours 02-05", "morning": "Morning 06-08"}

def save(fig, name):
    fig.savefig(FIG / f"{name}.png", dpi=150, bbox_inches="tight"); fig.savefig(FIG / f"{name}.svg", bbox_inches="tight")
    plt.close(fig)

def fig_cumulative():
    c = pd.read_csv(RES / "survival_cumulative_by_bin_full.csv").set_index("bin")
    hrs = [0, 1 / 6, 1, 3, 6, 12, 24]
    cols = ["cum_buy_pct_0.166667h", "cum_buy_pct_1h", "cum_buy_pct_3h", "cum_buy_pct_6h", "cum_buy_pct_12h", "cum_buy_pct_24h"]
    fig, ax = plt.subplots(figsize=(7, 4.2))
    for b in ["day", "evening", "late", "small", "morning"]:
        ax.plot(hrs, [0] + c.loc[b, cols].tolist(), marker="o", ms=4, color=COL[b], label=LAB[b], lw=2 if b in ("day", "late") else 1.3)
    ax.axvline(1, color="k", lw=.6, ls=":")
    ax.set_xlabel("Hours since first view"); ax.set_ylabel("Cumulative purchase rate (%)")
    ax.set_title("Test set: cumulative purchase after first view, by hour of first view")
    ax.set_xticks([0, 1, 3, 6, 12, 24]); ax.legend(fontsize=8, loc="lower right"); ax.grid(alpha=.2)
    save(fig, "04_cumulative_purchase_by_bin")

def fig_piecewise():
    m = pd.read_csv(RES / "survival_models_full.csv")
    terms = ["evening", "late", "small", "morning"]
    fig, ax = plt.subplots(figsize=(7, 3.8))
    for j, (mod, lab, mk) in enumerate([("M2_controls", "0-24h (M2)", "o"), ("PW_0_1h", "0-1h", "s"), ("PW_1h_24h", "1-24h (at risk at 1h)", "^")]):
        r = m[(m.model == mod)].set_index("term").loc[terms]
        y = np.arange(len(terms)) + (j - 1) * .22
        ax.errorbar(r.hr, y, xerr=[r.hr - r.hr_ci_low, r.hr_ci_high - r.hr], fmt=mk, capsize=3, label=lab)
    ax.axvline(1, color="k", lw=.8)
    ax.set_yticks(range(len(terms))); ax.set_yticklabels([LAB[t] for t in terms])
    ax.set_xlabel("Hazard ratio vs day 09-17 (95% CI, user-clustered)")
    ax.set_title("Cox models fitted on train (30.8M episodes)"); ax.legend(fontsize=8); ax.grid(axis="x", alpha=.2)
    save(fig, "05_cox_piecewise_hr")

def fig_cf():
    p = RES / "causal_forest_full.json"
    if not p.exists():
        return False
    cf = json.loads(p.read_text())["test"]
    g = pd.DataFrame(cf["gates"])
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(10, 4))
    a1.errorbar(g.quintile, g.coef, yerr=1.96 * g.se, fmt="o", capsize=4, color="#7C3A47", label="GATES (residual regression)")
    a1.plot(g.quintile, g.tau_hat_mean, "x--", color="gray", label="mean predicted CATE")
    a1.axhline(cf["dml_ate"]["coef"], color="#7C3A47", ls=":", lw=1, label="test ATE")
    a1.axhline(0, color="k", lw=.8)
    a1.set_xlabel("Quintile of predicted CATE (1 = most negative)"); a1.set_ylabel("Late vs day effect on 1h purchase (pp)")
    a1.set_title("(a) Test set GATES"); a1.legend(fontsize=8); a1.grid(alpha=.2)
    q = pd.read_csv(RES / "cate_quantiles_full.csv")
    q = q[(q["quantile"] >= .01) & (q["quantile"] <= .99)]
    a2.plot(q.cate_pp, q["quantile"], color="#7C3A47")
    a2.axvline(0, color="k", lw=.8)
    a2.set_xlabel("Predicted CATE (pp)"); a2.set_ylabel("Cumulative share of test episodes")
    a2.set_title("(b) Distribution of predicted CATE (1st-99th pct)"); a2.grid(alpha=.2)
    save(fig, "06_causal_forest_gates")
    return True

fig_cumulative(); fig_piecewise()
print("cf figure", fig_cf())


## 15 · Figures 07–08 (phase 2)

Fig 07 purchase hazard by hour × window; fig 08 cold vs hot item night loss.


In [ ]:
# ---- figures 07-08 ----
def fig07():
    m = pd.read_csv(RES / "survival_hourly_models_full.csv")
    c = pd.read_csv(RES / "survival_hourly_cumulative_full.csv").set_index("hour")

    fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.6))
    ax = axes[0]
    hours = np.arange(24)
    for win, col in [("w1h", "#7C3A47"), ("w3h", "#C98A95"), ("w24h", "#403737")]:
        sub = m[(m.model == f"M2_hour_{win}") & (m.term.str.startswith("h"))].set_index("term")
        hr = [1.0 if h == 14 else sub.loc[f"h{h:02d}", "hr"] for h in hours]
        lo = [np.nan if h == 14 else sub.loc[f"h{h:02d}", "hr_ci_low"] for h in hours]
        hi = [np.nan if h == 14 else sub.loc[f"h{h:02d}", "hr_ci_high"] for h in hours]
        ax.errorbar(hours, hr, yerr=[np.array(hr) - np.array(lo), np.array(hi) - np.array(hr)],
                    fmt="o-", ms=3, lw=1, capsize=2, color=col,
                    label={"w1h": "buy within 1h", "w3h": "buy within 3h", "w24h": "buy within 24h"}[win])
    ax.axhline(1.0, color="grey", ls="--", lw=0.8)
    ax.axvline(14, color="grey", ls=":", lw=0.8)
    ax.set_xticks(hours); ax.set_xticklabels([f"{h}" for h in hours], fontsize=7)
    ax.set_yscale("log"); ax.set_ylim(0.45, 1.45)
    ax.set_xlabel("hour of first view"); ax.set_ylabel("hazard ratio vs 14:00 (log scale)")
    ax.set_title("A. Purchase hazard by hour and window\n(Cox M2, train buckets 0-59, 30.8M episodes)")
    ax.legend(fontsize=8, frameon=False)
    ax = axes[1]
    for w, col, lab in [(1, "#7C3A47", "within 1h"), (3, "#C98A95", "within 3h"), (24, "#403737", "within 24h")]:
        ax.plot(hours, c[f"cum_buy_pct_{w:g}h"], "o-", ms=3, lw=1, color=col, label=lab)
    ax.axvline(14, color="grey", ls=":", lw=0.8)
    ax.set_xticks(hours); ax.set_xticklabels([f"{h}" for h in hours], fontsize=7)
    ax.set_xlabel("hour of first view"); ax.set_ylabel("cumulative purchase (%)")
    ax.set_title("B. Cumulative purchase by hour and window\n(test buckets 80-99)")
    ax.legend(fontsize=8, frameon=False)
    fig.tight_layout()
    fig.savefig(FIG / "07_survival_hourly.png", dpi=150)
    fig.savefig(FIG / "07_survival_hourly.svg", bbox_inches="tight")
    plt.close(fig)

def fig08():
    d = pd.read_csv(RES / "cold_hot_items_descriptive.csv")
    f = pd.read_csv(RES / "cold_hot_items_fe_late_summary.csv")
    xl = [("-1\n(viewed\nonce)", -1)] + [(str(i), i) for i in range(10)]
    fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.4))
    xs = np.arange(11)
    ax = axes[0]
    for col, lab, ls in [("day_y1_pct", "day, buy within 1h", "-"), ("late_y1_pct", "late, buy within 1h", "--"),
                         ("day_y2_pct", "day, buy within 24h", "-"), ("late_y2_pct", "late, buy within 24h", "--")]:
        ax.plot(xs, d[col], ls, marker="o", ms=3, label=lab,
                color={"day_y1_pct": "#403737", "late_y1_pct": "#7C3A47",
                       "day_y2_pct": "#1b5e20", "late_y2_pct": "#7C3A47"}[col])
    ax.set_xticks(xs); ax.set_xticklabels([l for l, _ in xl], fontsize=8)
    ax.set_ylabel("purchase rate (%)"); ax.set_xlabel("item popularity decile (cold -> hot)")
    ax.set_title("A. Conversion by item popularity\n(day vs late, raw)")
    ax.legend(fontsize=7.5, frameon=False)
    ax = axes[1]
    y1 = f[f.outcome == "y1"]; y2 = f[f.outcome == "y2"]
    ax.errorbar(np.arange(10), y1.coef, yerr=1.96 * y1.se, fmt="o-", color="#7C3A47", ms=4, capsize=3, label="buy within 1h")
    ax.errorbar(np.arange(10), y2.coef, yerr=1.96 * y2.se, fmt="s-", color="#D8A7AF", ms=4, capsize=3, label="buy within 24h")
    ax.axhline(0, color="grey", ls="--", lw=0.8)
    ax.set_xticks(np.arange(10)); ax.set_xticklabels([str(i) for i in range(10)], fontsize=8)
    ax.set_xlabel("item popularity decile (cold -> hot)"); ax.set_ylabel("late vs day (pp, FE-adjusted)")
    ax.set_title("B. Night loss by popularity\n(fixed effects, user+item+day)")
    ax.legend(fontsize=7.5, frameon=False)
    ax = axes[2]
    ax.plot(np.arange(10), y1.relative_pct, "o-", color="#7C3A47", ms=4, label="buy within 1h")
    ax.plot(np.arange(10), y2.relative_pct, "s-", color="#D8A7AF", ms=4, label="buy within 24h")
    ax.axhline(-24, color="grey", ls=":", lw=0.8)
    ax.set_xticks(np.arange(10)); ax.set_xticklabels([str(i) for i in range(10)], fontsize=8)
    ax.set_xlabel("item popularity decile (cold -> hot)")
    ax.set_ylabel("night loss as % of day rate")
    ax.set_title("C. Relative night loss by popularity\n(FE coef / day reference mean)")
    ax.legend(fontsize=7.5, frameon=False)
    fig.tight_layout()
    fig.savefig(FIG / "08_cold_hot_items.png", dpi=150)
    fig.savefig(FIG / "08_cold_hot_items.svg", bbox_inches="tight")
    plt.close(fig)

fig07(); fig08()
print("figures 07, 08 written")


## 16 · Figures 09–11 (explanatory + category)

Fig 09 what one data row looks like (events → episode); fig 10 category long-tail; fig 11 night penalty by category.


In [ ]:
# ---- figure 09: what one row of the data looks like ----
import matplotlib.dates as mdates
from datetime import datetime, timezone, timedelta

SH = timezone(timedelta(hours=8))
CJK = FontProperties(fname="/Library/Fonts/Arial Unicode.ttf")
_MAROON, _MID, _ROSE = "#7C3A47", "#A85D68", "#C98A95"
_PALE, _LINE, _CHAR, _GRAY = "#E7DADD", "#B9ABAE", "#4A4545", "#9A8F8F"
_GREY_PT = "#B3A6AA"

USER, ITEM = 77, 4834883

ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["row_no", "user_id", "item_id", "category_id", "behavior", "ts"])
ev = ev[ev.user_id == USER].sort_values("ts").reset_index(drop=True)
ev["dt"] = [datetime.fromtimestamp(t, SH) for t in ev.ts]
B = {1: "pv", 2: "fav", 3: "cart", 4: "buy"}
STYLE = {"pv": (_GREY_PT, 22, "o"), "fav": (_ROSE, 55, "s"),
         "cart": (_MID, 60, "^"), "buy": (_MAROON, 190, "*")}

ep = ds.dataset(DATA / "episodes_hour.parquet").to_table(filter=(ds.field("user_id") == USER) & (ds.field("item_id") == ITEM)).to_pandas().iloc[0]

fig = plt.figure(figsize=(12.6, 8.9))

def _mk(ax):
    return lambda x, y, t, **kw: ax.text(x, y, t, transform=ax.transAxes, **kw)
fig.patch.set_facecolor("white")

ax1 = fig.add_axes([0.05, 0.705, 0.92, 0.265]); ax1.axis("off")
t1 = _mk(ax1)
t1(0, 1.02, "1. raw data: one row = one user action", fontsize=15.5, color=_MAROON, fontweight="bold", va="top")
t1(0, 0.845, "Tianchi 649 UserBehavior.csv (100,095,182 rows x 5 cols)", fontsize=9.5, color=_GRAY, va="top")
lines = ["user_id,item_id,category_id,behavior_type,timestamp"]
win = ev[(ev.ts >= 1512191700) & (ev.ts <= 1512192100)]
for r in win.itertuples():
    lines.append(f"{r.user_id},{r.item_id},{r.category_id},{B[r.behavior]},{r.ts}")
t1(0.0, 0.70, "\n".join(lines), family="monospace", fontsize=10.5, va="top", color=_CHAR, linespacing=1.75,
     bbox=dict(boxstyle="round,pad=0.6", facecolor="#FBF7F8", edgecolor=_PALE))
t1(0.46, 0.70,
     "timestamp is Unix seconds: 1512191890 -> 2017-12-02 13:18:10\n"
     "behavior_type has 4 kinds: pv / fav / cart / buy\n"
     "no price, quantity, dwell time",
     fontsize=10, color=_CHAR, va="top", linespacing=1.75)

ax2 = fig.add_axes([0.05, 0.395, 0.92, 0.245])
lanes = {"buy": 0, "cart": 1, "fav": 2, "pv": 3}
for r in ev.itertuples():
    c, s, m = STYLE[B[r.behavior]]
    ax2.scatter(r.dt, lanes[B[r.behavior]], s=s, marker=m, color=c, zorder=3, edgecolors="white", linewidths=0.7)
foc = ev[ev.item_id == ITEM]
ax2.scatter([d.to_pydatetime() for d in foc.dt], [lanes["buy"]] + [lanes["pv"]] * (len(foc) - 1),
            s=260, facecolors="none", edgecolors=_MAROON, linewidths=1.4, zorder=4)
ax2.set_yticks(list(lanes.values()))
ax2.set_yticklabels(["buy", "cart", "fav", "pv"], fontsize=10)
ax2.set_ylim(-1.15, 3.6)
ax2.set_xlim(ev.dt.min() - pd.Timedelta(hours=8), ev.dt.max() + pd.Timedelta(hours=8))
ax2.xaxis.set_major_formatter(mdates.DateFormatter("%m-%d"))
ax2.xaxis.set_major_locator(mdates.DayLocator())
ax2.tick_params(axis="x", labelsize=9, colors=_CHAR, length=3)
ax2.grid(axis="x", color="#F0E9EA", lw=0.6)
for sp in ("top", "right", "left"):
    ax2.spines[sp].set_visible(False)
ax2.spines["bottom"].set_color(_LINE)
ax2.text(ev.dt.min(), -0.95,
         "circled = same item 4834883: first view 13:18 -> buy 13:20 (2m10s apart)",
         fontsize=9.5, color=_MAROON, ha="left", va="center")
ax2.set_title("2. same user, 20 actions over 9 days (swim lanes)", fontsize=13.5, color=_MAROON, fontweight="bold", loc="left", pad=12)

ax3 = fig.add_axes([0.05, 0.035, 0.92, 0.30]); ax3.axis("off")
t3 = _mk(ax3)
t3(0, 1.02, "3. this pair collapses to 1 episode row", fontsize=15.5, color=_MAROON, fontweight="bold", va="top")
fields = [("user_id", "77", "who"), ("item_id", "4834883", "which item"),
          ("t0", "13:18:10", "first-view instant"), ("hour", "13", "hour of t0"), ("day", "7", "day of t0"),
          ("buy_delay_s", "129", "seconds until buy"), ("hist_pv24", "9", "his pvs in prior 24h"),
          ("pop24", "71", "item pvs in prior 24h")]
t3(0.00, 0.86, "field", fontsize=9, color=_GRAY, va="top")
t3(0.155, 0.86, "value", fontsize=9, color=_GRAY, va="top")
t3(0.275, 0.86, "meaning", fontsize=9, color=_GRAY, va="top")
for i, (f, v, m) in enumerate(fields):
    y = 0.745 - i * 0.093
    t3(0.00, y, f, family="monospace", fontsize=9.5, color=_CHAR, va="top")
    t3(0.155, y, v, family="monospace", fontsize=9.5, color=_MAROON, va="top")
    t3(0.275, y, m, fontsize=9.5, color=_CHAR, va="top")
ax3.plot([0.575, 0.575], [0.05, 0.88], color=_PALE, lw=1, transform=ax3.transAxes)
t3(0.60, 0.86, "outcomes derive from buy_delay_s", fontsize=9.5, color=_CHAR, va="top")
t3(0.60, 0.735, "buy within 1h = (0 <= 129 < 3600) -> 1", fontsize=9.5, color=_MAROON, va="top")
t3(0.60, 0.645, "buy within 24h = (0 <= 129 < 86400) -> 1", fontsize=9.5, color=_MAROON, va="top")
t3(0.60, 0.50, "if no buy in 9 days: buy_delay_s = -1,", fontsize=9, color=_GRAY, va="top")
t3(0.60, 0.425, "both outcomes = 0.", fontsize=9, color=_GRAY, va="top")
t3(0.60, 0.28, "so: 100M raw actions -> 51.3M episodes", fontsize=9.5, color=_CHAR, va="top")

fig.savefig(FIG / "09_data_rows.png", dpi=150, bbox_inches="tight", facecolor="white")
print("saved", FIG / "09_data_rows.png")


In [ ]:
# ---- figure 10: category long-tail overview ----
apply()
CJK = FontProperties(fname="/Library/Fonts/Arial Unicode.ttf")

ev = pd.read_parquet(PROC / "events_clean.parquet", columns=["category_id"])
vc = ev.category_id.value_counts()
tot, ncat = len(ev), vc.size
share = vc / tot * 100
cum = share.cumsum()

fig, (a1, a2) = plt.subplots(1, 2, figsize=(12.4, 4.6), gridspec_kw={"width_ratios": [1.05, 1]})
top = share.head(12)[::-1]
ypos = np.arange(len(top))
a1.barh(ypos, top.values, height=.62, color=[MAROON if i >= 10 else MAROON_MID for i in range(len(top))])
a1.set_yticks(ypos); a1.set_yticklabels([f"{int(i)}" for i in top.index], fontsize=8.5, color=CHAR)
for y, v in zip(ypos, top.values):
    a1.text(v + 0.06, y, f"{v:.2f}%", va="center", fontsize=8, color=CHAR)
a1.set_xlim(0, max(top) * 1.22)
a1.set_xlabel("share of all events (%)", color=CHAR)
a1.set_title("Top 12 categories (anonymised IDs)", fontsize=11, color=CHAR)
a1.grid(axis="x")

a2.plot(np.arange(1, ncat + 1), cum.values, color=MAROON, lw=1.6)
a2.set_xscale("log")
a2.set_ylim(0, 101)
for k in [10, 100, 1000]:
    a2.axvline(k, color=PINK, lw=.8, ls=":")
    a2.plot([k], [cum.iloc[k - 1]], "o", color=MAROON, ms=4)
    a2.annotate(f"top {k}: {cum.iloc[k-1]:.0f}%", xy=(k, cum.iloc[k - 1]),
                xytext=(k * 1.15, cum.iloc[k - 1] - 13), fontsize=8.5, color=MAROON,
                arrowprops=dict(arrowstyle="-", color=MAROON, lw=.6))
a2.set_xlabel("category rank (log scale)", color=CHAR)
a2.set_ylabel("cumulative share of events (%)", color=CHAR)
a2.set_title(f"Long tail: {ncat:,} categories in total", fontsize=11, color=CHAR)
a2.grid(True)

fig.tight_layout()
fig.savefig(FIG / "10_categories.png", dpi=150, bbox_inches="tight", facecolor="white")
print("saved", FIG / "10_categories.png", "| categories:", ncat)


In [ ]:
# ---- figure 11: night penalty by category (24h window HR) ----
apply()
CJK = FontProperties(fname="/Library/Fonts/Arial Unicode.ttf")

s = pd.read_csv(RES / "category_survival_summary_full.csv")
s = s.sort_values("late_HR_w24h").reset_index(drop=True)

fig, ax = plt.subplots(figsize=(12.2, 5.6))
labels = [("OTHER (9,400 small cats)" if g == "OTHER" else str(g)) for g in s.label]
ypos = np.arange(len(s))[::-1]
colors = [PINK if g == "OTHER" else MAROON for g in s.label]
ax.barh(ypos, s.late_HR_w24h, height=.66, color=colors, zorder=3)
ax.errorbar(s.late_HR_w24h, ypos,
            xerr=[s.late_HR_w24h - s.late_HRlo_w24h, s.late_HRhi_w24h - s.late_HR_w24h],
            fmt="none", ecolor=CHAR, capsize=2, lw=.8, zorder=4)
for y, v in zip(ypos, s.late_HR_w24h):
    ax.text(v + .008, y, f"{v:.2f}", va="center", fontsize=7.5, color=CHAR)
ax.axvline(1.0, color="#4A4545", ls="--", lw=1.1)
ax.set_yticks(ypos)
ax.set_yticklabels(labels, fontsize=8.5, color=CHAR, fontproperties=CJK)
ax.set_xlim(0.40, 1.06)
ax.set_xlabel("late vs day hazard ratio (24h window, HR < 1 = fewer late buys)", color=CHAR, fontsize=9)
ax.set_title("Night penalty by category: 21 groups (top-20 + other)", fontsize=12, color=CHAR, pad=12)
ax.grid(axis="x")
ax.text(0.985, 0.35, "HR = 1\n(parity)", fontsize=7.5, color=GRAY, ha="right", va="bottom")

fig.tight_layout()
fig.savefig(FIG / "11_category_night_penalty.png", dpi=150, bbox_inches="tight", facecolor="white")
print("saved", FIG / "11_category_night_penalty.png")


## Verification summary

Independent re-implementations and cross-checks, all of which passed.


In [ ]:
# ---- verification summary ----
v  = json.loads((RES / "verification.json").read_text())
fv = json.loads((RES / "features_verification.json").read_text())
fc = json.loads((RES / "fastcox_verification.json").read_text())
print("episode rebuild, 400 users       : field mismatches =", v["rebuild"]["field_mismatches"])
print("pop24 recount, 300 episodes      : mismatches =", v["pop24_mismatches_of_300"])
print("FE vs explicit-dummy OLS         : max coef diff =", v["fe_vs_dummies"]["max_abs_coef_diff"])
print("feature recount, 300 episodes    :", fv["feature_mismatches_of_300"])
print("train/val/test user overlap      :", fv["split_user_overlap"])
print("FastCox vs statsmodels           : max coef diff =", fc["vs_statsmodels"]["max_abs_coef_diff"])


## Results summary

| Method | Key result |
|---|---|
| FE-LPM (M2) | late vs day Y1 −0.179 pp, Y2 −0.307 pp; small Y1 −0.238 pp, Y2 −0.230 pp (Holm p<1e-15) |
| Survival (Cox) | late HR 0.705 (→ "fewer", 1–24h HR 0.635); small HR 0.743 (→ "later", 1–24h HR 0.975) |
| Causal Forest | ATE −0.181 pp (matches FE); BLP β 1.26; GATES Q5−Q1 +0.39 pp; heterogeneity is mostly scale (relative −17% to −26%) |
| Phase 2 A (hourly) | 09–11 highest across windows; 23:00 24h HR 0.63 lowest; 00–02 "later/overnight"; 04–07 lowest immediate |
| Phase 2 B (cold/hot) | absolute night loss largest for cold items (−0.44 → −0.08 pp); relative −38% → −19% |
| Category | 21 groups all have late 24h HR < 1 (0.58–0.81); category importance in CF ≈ 0 (6.5e-5) |

## Conclusion

The data do not support "late night = impulse buying"; they support "late night = idle browsing". For business: late-night traffic converts less both immediately and next-day, so time-of-day bidding can down-weight late night; small-hours traffic is mostly delayed conversion, so evaluate on 24 h conversion.

## Limitations (keep in the report)

Observational, not experimental; 9 days, no price; "first visible view" ≠ true first view; the flat 1–6h night curve partly reflects sleep.

## Reading

- Chinese findings: `FINDINGS_zh.md`, `FINDINGS_PHASE2_zh.md`
- Plans & deviations: `ANALYSIS_PLAN.md`, `ANALYSIS_PLAN_ADVANCED.md`, `ANALYSIS_PLAN_PHASE2.md`, `POST_PLAN_DEVIATIONS.md`
- Results: `results/`; figures: `figures/01–11`
